# NSSP — notebook utama: Table 1–8 dan Supplementary S1–S33
Notebook ini membangun ulang semua tabel naskah *Selecting Spatial Connectivity Structures for Spatial Econometric Analysis* dari data mentah, lalu **memeriksa hasilnya secara otomatis** terhadap angka di naskah dan hasil addon5 (`CHECK_vs_manuscript_and_addon5.csv`).

**Input (Add Input):** dataset panel 2020–2024 (wajib); `profile_presample_2019` (untuk S2b, S22, S23a); `geo_jateng_kaggle` (centroid untuk KNN-4 dan inverse distance: Table 5, S15, set diperluas). Matriks W yang dibekukan sudah tertanam.

**Cara pakai:** uji dulu dengan `QUICK = True` (≈ 2–3 menit), lalu `QUICK = False` → *Save & Run All* (≈ 20–40 menit di CPU). Semua tabel tersimpan di `/kaggle/working/nssp_main_results/` dan `nssp_main_results.zip`.

Table S24/S24b, S34–S36 dihasilkan oleh notebook addon6 dan addon7. Table S18b (Hausman spasial) dan S25 (angka dari estimasi lama) tidak dihitung ulang di sini.

In [ ]:
# Sel 1 — Konfigurasi
import os, glob, json
INPUT_DIR = "/kaggle/input"
OUT = "/kaggle/working"
RES = os.path.join(OUT, "nssp_main_results"); os.makedirs(RES, exist_ok=True)

def _find(pattern, exclude=()):
    hits = [h for h in glob.glob(os.path.join(INPUT_DIR, "**", pattern), recursive=True)
            if not any(e in h for e in exclude)]
    return sorted(hits)[0] if hits else None

# --- file input (otomatis; isi manual jika perlu) ---
DATA_PATH = _find("*panel*2020*2024*.csv") or _find("panel*.csv")
PRESAMPLE_PATH = _find("*2019*.csv") or _find("*presample*.csv") or _find("*2019*.xlsx")
GEO_PATH = (_find("*.shp") or _find("*.geojson") or _find("*.gpkg")
            or _find("*geo*.csv") or _find("*centroid*.csv"))

# --- matriks W yang dibekukan (identik dengan W_*.csv di nssp_addon5) ---
W_EMBED = {'queen': ',3301,3302,3303,3304,3305,3306,3307,3308,3309,3310,3311,3312,3313,3314,3315,3316,3317,3318,3319,3320,3321,3322,3323,3324,3325,3326,3327,3328,3329,3371,3372,3373,3374,3375,3376\n3301,0.0,0.3333333333333333,0.0,0.0,0.3333333333333333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.3333333333333333,0.0,0.0,0.0,0.0,0.0,0.0\n3302,0.14285714285714285,0.0,0.14285714285714285,0.14285714285714285,0.14285714285714285,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.14285714285714285,0.14285714285714285,0.14285714285714285,0.0,0.0,0.0,0.0,0.0,0.0\n3303,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3304,0.0,0.16666666666666666,0.16666666666666666,0.0,0.16666666666666666,0.0,0.16666666666666666,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.16666666666666666,0.16666666666666666,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3305,0.2,0.2,0.0,0.2,0.0,0.2,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3306,0.0,0.0,0.0,0.0,0.3333333333333333,0.0,0.3333333333333333,0.3333333333333333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3307,0.0,0.0,0.0,0.14285714285714285,0.14285714285714285,0.14285714285714285,0.0,0.14285714285714285,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.14285714285714285,0.14285714285714285,0.14285714285714285,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3308,0.0,0.0,0.0,0.0,0.0,0.14285714285714285,0.14285714285714285,0.0,0.14285714285714285,0.14285714285714285,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.14285714285714285,0.14285714285714285,0.0,0.0,0.0,0.0,0.0,0.0,0.14285714285714285,0.0,0.0,0.0,0.0,0.0\n3309,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.125,0.0,0.125,0.125,0.0,0.125,0.125,0.125,0.0,0.0,0.0,0.0,0.0,0.0,0.125,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.125,0.0,0.0,0.0,0.0\n3310,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.3333333333333333,0.3333333333333333,0.0,0.3333333333333333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3311,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.2,0.0,0.2,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.0,0.0,0.0,0.0\n3312,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3313,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.0,0.2,0.2,0.0,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.0,0.0,0.0,0.0\n3314,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.3333333333333333,0.0,0.0,0.0,0.3333333333333333,0.0,0.3333333333333333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3315,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.14285714285714285,0.0,0.0,0.0,0.0,0.14285714285714285,0.0,0.14285714285714285,0.0,0.14285714285714285,0.14285714285714285,0.0,0.14285714285714285,0.14285714285714285,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3316,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.3333333333333333,0.0,0.3333333333333333,0.3333333333333333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3317,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3318,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.2,0.2,0.0,0.2,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3319,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.25,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3320,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.3333333333333333,0.3333333333333333,0.0,0.3333333333333333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3321,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.0,0.0,0.0,0.2,0.2,0.0,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.0,0.0\n3322,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.125,0.125,0.0,0.0,0.0,0.0,0.0,0.125,0.0,0.0,0.0,0.0,0.0,0.125,0.0,0.125,0.125,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.125,0.125,0.0,0.0\n3323,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3324,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.2,0.0,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.0,0.0\n3325,0.0,0.0,0.0,0.2,0.0,0.0,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.0,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.0\n3326,0.0,0.0,0.2,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.0,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.0\n3327,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3328,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.25\n3329,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.25\n3371,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3372,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.3333333333333333,0.0,0.3333333333333333,0.0,0.3333333333333333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3373,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3374,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.3333333333333333,0.3333333333333333,0.0,0.3333333333333333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3375,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3376,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.5,0.0,0.0,0.0,0.0,0.0,0.0\n', 'knn4': ',3301,3302,3303,3304,3305,3306,3307,3308,3309,3310,3311,3312,3313,3314,3315,3316,3317,3318,3319,3320,3321,3322,3323,3324,3325,3326,3327,3328,3329,3371,3372,3373,3374,3375,3376\n3301,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0\n3302,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0\n3303,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3304,0.0,0.0,0.25,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3305,0.0,0.0,0.25,0.25,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3306,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0\n3307,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0\n3308,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.0,0.0\n3309,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0\n3310,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0\n3311,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0\n3312,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0\n3313,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0\n3314,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0\n3315,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.25,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3316,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3317,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3318,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3319,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.25,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3320,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.25,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3321,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0\n3322,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0\n3323,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0\n3324,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0\n3325,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0\n3326,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0\n3327,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.25,0.0\n3328,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.25\n3329,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.25\n3371,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0\n3372,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3373,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0\n3374,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0\n3375,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3376,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0\n', 'econ': ',3301,3302,3303,3304,3305,3306,3307,3308,3309,3310,3311,3312,3313,3314,3315,3316,3317,3318,3319,3320,3321,3322,3323,3324,3325,3326,3327,3328,3329,3371,3372,3373,3374,3375,3376\n3301,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.25,0.0,0.0\n3302,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.25,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3303,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3304,0.0,0.0,0.25,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3305,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3306,0.0,0.0,0.0,0.25,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3307,0.0,0.0,0.25,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3308,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3309,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3310,0.0,0.25,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3311,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0\n3312,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3313,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.25,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3314,0.0,0.25,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3315,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3316,0.0,0.0,0.0,0.25,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3317,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3318,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.25,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3319,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3320,0.0,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3321,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3322,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3323,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3324,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0\n3325,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3326,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3327,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25\n3328,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.25\n3329,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.25,0.0,0.0,0.0,0.0\n3371,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.25\n3372,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0\n3373,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.25,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0\n3374,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.0,0.0,0.0\n3375,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0.25,0.0,0.25,0.0,0.0,0.25\n3376,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.25,0.0,0.25,0.0,0.0,0.0,0.25,0.0\n'}
W_PATHS = {}
for k, txt in W_EMBED.items():
    p = _find(f"W_{k}.csv")
    if p is None:
        p = os.path.join(OUT, f"W_{k}_embedded.csv")
        with open(p, "w") as f:
            f.write(txt)
    W_PATHS[k] = p

# --- nama kolom: None = deteksi otomatis di Sel 2 ---
COL_ID, COL_YEAR, COL_Y = None, None, None
COL_X = {"manuf": None, "lgrdp": None, "tpt": None, "sanit": None}   # urutan = naskah
COL_HDI, COL_TPAK = None, None

YEARS = [2020, 2021, 2022, 2023, 2024]
SUB_YEARS = [2022, 2023, 2024]
SEED = 20260101
N_DRAWS = 5000            # draws untuk interval impacts (naskah: 5,000)
N_MIX = 20000             # draws untuk model averaging (naskah: 20,000)
BAYES_S = 4000            # draws Bayes statis (elpd)
REFIT_ITER, REFIT_BURN = 3000, 1000   # exact LOO refit (data augmentation)
B_EMP, MAX_ITER, TOL = 499, 25, 1e-3  # bootstrap dinamis
BAYES_ITER_EMP, BAYES_BURN_EMP = 15000, 5000   # Bayes dinamis (Table S30)
MC_RHO_REPS = 500         # Table S5
MC_W_REPS = 300           # Table S32
RUN_MC = True             # False = lewati Table S5 dan S32
N_JOBS = -1
QUICK = False             # True = uji cepat (~2 menit)

# --- nilai rujukan (addon5 dan naskah) untuk pemeriksaan otomatis ---
REF_CSV = {"E5b": "W,FE,variable,effect,estimate,lwr95,upr95\nQueen contiguity,ind,manuf,direct,0.018627213254332777,-0.005866002718160742,0.043174259786308164\nQueen contiguity,ind,lgrdp,direct,-6.010548703094613,-9.023280820452644,-3.095497359462599\nQueen contiguity,ind,tpt,direct,-0.026545862790365533,-0.0934373014213624,0.03879321423712614\nQueen contiguity,ind,sanit,direct,-0.0347798096056473,-0.053377153011859683,-0.01620816234552156\nQueen contiguity,ind,manuf,indirect,0.03086676893312065,-0.010906669942490167,0.07146815515919687\nQueen contiguity,ind,lgrdp,indirect,-1.9601027454753126,-5.720987536421057,1.914045989303407\nQueen contiguity,ind,tpt,indirect,-0.2818662656250519,-0.5079619556313751,-0.08277149687596291\nQueen contiguity,ind,sanit,indirect,-0.07935980361941988,-0.1377268313317647,-0.030041518056928774\nQueen contiguity,ind,manuf,total,0.04949398218745343,0.009447951995916479,0.0887599868604142\nQueen contiguity,ind,lgrdp,total,-7.970651448569925,-10.905599681498911,-5.016485999919307\nQueen contiguity,ind,tpt,total,-0.30841212841541743,-0.5625373430211221,-0.08393223675556384\nQueen contiguity,ind,sanit,total,-0.11413961322506719,-0.18373841095837953,-0.05466555830134281\nQueen contiguity,tw,manuf,direct,0.004891937466077055,-0.02067437872596222,0.029962676794816293\nQueen contiguity,tw,lgrdp,direct,-5.127253053525115,-8.181473098686968,-1.9697165962776733\nQueen contiguity,tw,tpt,direct,-0.020041983135299445,-0.07505978927201679,0.03596761434435169\nQueen contiguity,tw,sanit,direct,-0.03250795039930428,-0.047826458052498445,-0.016842369180238792\nQueen contiguity,tw,manuf,indirect,-0.015014755880824033,-0.07352168381701178,0.04036186713792327\nQueen contiguity,tw,lgrdp,indirect,1.8613780026337055,-3.960194388897601,7.47470264555875\nQueen contiguity,tw,tpt,indirect,-0.11506136525157579,-0.223299076349139,-0.008866802502051229\nQueen contiguity,tw,sanit,indirect,-0.05974132056032269,-0.08699561337973873,-0.03453179965329935\nQueen contiguity,tw,manuf,total,-0.010122818414746979,-0.07720165888126146,0.05171264291055534\nQueen contiguity,tw,lgrdp,total,-3.265875050891409,-9.64385961298105,3.2045086226029653\nQueen contiguity,tw,tpt,total,-0.13510334838687524,-0.24880601582610218,-0.026964843444826456\nQueen contiguity,tw,sanit,total,-0.09224927095962697,-0.12250062811055604,-0.0642195310795964\n4-nearest neighbours (great circle),ind,manuf,direct,0.006996435086909538,-0.017262671596845785,0.031260882787024276\n4-nearest neighbours (great circle),ind,lgrdp,direct,-6.327154952415418,-9.201196579900015,-3.4692500469642544\n4-nearest neighbours (great circle),ind,tpt,direct,-0.03279119742724593,-0.10041078200023859,0.034127427559441564\n4-nearest neighbours (great circle),ind,sanit,direct,-0.03322004132451429,-0.05215492986709802,-0.014268980670374925\n4-nearest neighbours (great circle),ind,manuf,indirect,0.04960495439730583,0.0028199452373170253,0.09509017338772843\n4-nearest neighbours (great circle),ind,lgrdp,indirect,-0.7651453399650716,-4.987473622527303,3.506950605913921\n4-nearest neighbours (great circle),ind,tpt,indirect,-0.1333390617465784,-0.3857655460464179,0.11180478466008426\n4-nearest neighbours (great circle),ind,sanit,indirect,-0.07563385689481547,-0.142784654313382,-0.011954701982681717\n4-nearest neighbours (great circle),ind,manuf,total,0.056601389484215364,0.01108608537982876,0.10197525301742932\n4-nearest neighbours (great circle),ind,lgrdp,total,-7.092300292380489,-10.849124474364295,-3.2327097242850567\n4-nearest neighbours (great circle),ind,tpt,total,-0.16613025917382432,-0.453547057326184,0.11459063828995875\n4-nearest neighbours (great circle),ind,sanit,total,-0.10885389821932977,-0.18540935806768938,-0.03499169009485402\n4-nearest neighbours (great circle),tw,manuf,direct,0.0053181111503915485,-0.02034573307478857,0.030874510480253185\n4-nearest neighbours (great circle),tw,lgrdp,direct,-5.1260515692919135,-8.50509579287718,-1.770941127545166\n4-nearest neighbours (great circle),tw,tpt,direct,-0.03910364282645517,-0.09428184296467981,0.014475945164760345\n4-nearest neighbours (great circle),tw,sanit,direct,-0.03173558037653362,-0.04839590637642787,-0.015207925184235938\n4-nearest neighbours (great circle),tw,manuf,indirect,0.03320963097941108,-0.03350998748407914,0.10153410550174927\n4-nearest neighbours (great circle),tw,lgrdp,indirect,4.9947469430154285,-2.0875783764237026,12.8100279792159\n4-nearest neighbours (great circle),tw,tpt,indirect,-0.09812271430809924,-0.22017517196591968,0.025571599009469252\n4-nearest neighbours (great circle),tw,sanit,indirect,-0.05355136759341335,-0.09004407406576959,-0.017051353936105886\n4-nearest neighbours (great circle),tw,manuf,total,0.038527742129802624,-0.03484116120512162,0.11642830456411075\n4-nearest neighbours (great circle),tw,lgrdp,total,-0.131304626276485,-8.72315833884394,9.180046622590094\n4-nearest neighbours (great circle),tw,tpt,total,-0.13722635713455442,-0.2736102943065083,-0.0016387024669693173\n4-nearest neighbours (great circle),tw,sanit,total,-0.08528694796994697,-0.12692466032357003,-0.0431904578931606\nEconomic k=4 (2020 profile),ind,manuf,direct,0.004598804399286194,-0.023875630115317178,0.0324304594127412\nEconomic k=4 (2020 profile),ind,lgrdp,direct,-3.536091031122794,-6.629528275866134,-0.3899634105098994\nEconomic k=4 (2020 profile),ind,tpt,direct,-0.02831544317814133,-0.09857863521987083,0.04229688627578201\nEconomic k=4 (2020 profile),ind,sanit,direct,-0.03146404757917211,-0.051886288726675105,-0.01074070469357935\nEconomic k=4 (2020 profile),ind,manuf,indirect,0.03889407473728988,-0.010108420491195742,0.0856661032820398\nEconomic k=4 (2020 profile),ind,lgrdp,indirect,-4.4510708178746405,-9.191394052963533,0.03126932382359851\nEconomic k=4 (2020 profile),ind,tpt,indirect,-0.16423836307852152,-0.4242298429603921,0.0702817760763294\nEconomic k=4 (2020 profile),ind,sanit,indirect,-0.03727271890787179,-0.12488295438726014,0.05996511552429404\nEconomic k=4 (2020 profile),ind,manuf,total,0.043492879136576075,-0.0063150246045012145,0.09307014994410394\nEconomic k=4 (2020 profile),ind,lgrdp,total,-7.9871618489974345,-12.296432142130982,-4.24664251802039\nEconomic k=4 (2020 profile),ind,tpt,total,-0.19255380625666285,-0.48396523983616696,0.07072365739043102\nEconomic k=4 (2020 profile),ind,sanit,total,-0.0687367664870439,-0.16764082176102435,0.04094447998394374\nEconomic k=4 (2020 profile),tw,manuf,direct,-0.0034871044989973274,-0.031931631712755565,0.024392161003016468\nEconomic k=4 (2020 profile),tw,lgrdp,direct,-3.9830973911462983,-7.435343525465023,-0.5557131591606994\nEconomic k=4 (2020 profile),tw,tpt,direct,-0.022392437553738372,-0.08679117553615243,0.03976307076323092\nEconomic k=4 (2020 profile),tw,sanit,direct,-0.03738097066965723,-0.0549525382096168,-0.019848450574961665\nEconomic k=4 (2020 profile),tw,manuf,indirect,-0.018391247897680065,-0.08428726552895748,0.040943810445973006\nEconomic k=4 (2020 profile),tw,lgrdp,indirect,-3.7384777548149604,-17.011821670482735,8.023877976803139\nEconomic k=4 (2020 profile),tw,tpt,indirect,-0.08642893210006611,-0.21199225571638647,0.0311498665988132\nEconomic k=4 (2020 profile),tw,sanit,indirect,-0.03870762211250135,-0.08160533719837423,0.004527317468548289\nEconomic k=4 (2020 profile),tw,manuf,total,-0.021878352396677393,-0.09335791959219183,0.04445889695886097\nEconomic k=4 (2020 profile),tw,lgrdp,total,-7.721575145961259,-22.319931027827018,5.2994753399432195\nEconomic k=4 (2020 profile),tw,tpt,total,-0.10882136965380448,-0.25034844817479585,0.018058477633243555\nEconomic k=4 (2020 profile),tw,sanit,total,-0.07608859278215858,-0.1244544299468827,-0.025728210088056896\nInverse distance (non-truncated),ind,manuf,direct,-0.0002513973548075995,-0.029957285835570786,0.024769616126188714\nInverse distance (non-truncated),ind,lgrdp,direct,-4.793593808527082,-8.007224283594011,-1.3933175406232143\nInverse distance (non-truncated),ind,tpt,direct,-0.07262327944881972,-0.21328658930514477,0.005780560977701879\nInverse distance (non-truncated),ind,sanit,direct,-0.05556997223905903,-0.11451272404583483,-0.033060082610167844\nInverse distance (non-truncated),ind,manuf,indirect,-0.09556403039263452,-0.5314800701878224,0.02408122850844153\nInverse distance (non-truncated),ind,lgrdp,indirect,2.034966339865946,-22.68335525026667,34.01108243282168\nInverse distance (non-truncated),ind,tpt,indirect,-1.1006466486622142,-5.379897248108119,0.34865353219645473\nInverse distance (non-truncated),ind,sanit,indirect,-0.788984275862064,-2.678223567476247,-0.2792729033419723\nInverse distance (non-truncated),ind,manuf,total,-0.09581542774744212,-0.5452805752177721,0.024815379984576138\nInverse distance (non-truncated),ind,lgrdp,total,-2.7586274686611363,-28.341701774474604,29.529297423168973\nInverse distance (non-truncated),ind,tpt,total,-1.173269928111034,-5.56956512074513,0.3350886156605982\nInverse distance (non-truncated),ind,sanit,total,-0.844554248101123,-2.7881204491824754,-0.3169040155046106\nInverse distance (non-truncated),tw,manuf,direct,0.004730027299261367,-0.020653336013716803,0.030041402681230026\nInverse distance (non-truncated),tw,lgrdp,direct,-4.853324712917931,-8.116446172004407,-1.6691930351212145\nInverse distance (non-truncated),tw,tpt,direct,-0.053572297096623825,-0.10715465819552701,-0.00031061362404700727\nInverse distance (non-truncated),tw,sanit,direct,-0.037657718116466725,-0.05367739905009686,-0.0222287582525178\nInverse distance (non-truncated),tw,manuf,indirect,0.020299489520012683,-0.176125950225382,0.20012451851747418\nInverse distance (non-truncated),tw,lgrdp,indirect,3.520380984182367,-11.399092795724446,21.298079442209843\nInverse distance (non-truncated),tw,tpt,indirect,-0.15362132432043757,-0.495085945065779,0.06763648843721443\nInverse distance (non-truncated),tw,sanit,indirect,-0.1666922978914433,-0.30188806264361334,-0.0952087091861956\nInverse distance (non-truncated),tw,manuf,total,0.02502951681927405,-0.17728784957236418,0.21132898804296066\nInverse distance (non-truncated),tw,lgrdp,total,-1.3329437287355639,-16.53747267158014,16.889656647975404\nInverse distance (non-truncated),tw,tpt,total,-0.20719362141706138,-0.5606025117350021,0.013367188551532528\nInverse distance (non-truncated),tw,sanit,total,-0.20435001600791003,-0.34060070789192,-0.13245294702300434\n\"Inverse distance, cut-off 69 km\",ind,manuf,direct,0.012706766367465074,-0.01296136504802751,0.03767705150520606\n\"Inverse distance, cut-off 69 km\",ind,lgrdp,direct,-6.577006893477869,-9.615148480121636,-3.4738221082791427\n\"Inverse distance, cut-off 69 km\",ind,tpt,direct,-0.04098362745286824,-0.10875548563771154,0.026326845372380288\n\"Inverse distance, cut-off 69 km\",ind,sanit,direct,-0.0402075814798069,-0.06000638792139151,-0.02112017849846745\n\"Inverse distance, cut-off 69 km\",ind,manuf,indirect,0.025306709676350653,-0.04000304058984147,0.07848722350356943\n\"Inverse distance, cut-off 69 km\",ind,lgrdp,indirect,-0.2593680132586753,-6.399533896152019,5.68874685248894\n\"Inverse distance, cut-off 69 km\",ind,tpt,indirect,-0.24255061780015272,-0.7017131173400101,0.12703869822315553\n\"Inverse distance, cut-off 69 km\",ind,sanit,indirect,-0.13736798389633145,-0.27414933089158,-0.029799293768229252\n\"Inverse distance, cut-off 69 km\",ind,manuf,total,0.038013476043815726,-0.025548110475736997,0.08901086360224493\n\"Inverse distance, cut-off 69 km\",ind,lgrdp,total,-6.836374906736545,-12.721921772490678,-0.8604831260813279\n\"Inverse distance, cut-off 69 km\",ind,tpt,total,-0.28353424525302096,-0.7886968265548941,0.11540120423104394\n\"Inverse distance, cut-off 69 km\",ind,sanit,total,-0.17757556537613836,-0.3241867320261973,-0.05936678079097138\n\"Inverse distance, cut-off 69 km\",tw,manuf,direct,0.011767044694953514,-0.013860976500602072,0.03679018557944014\n\"Inverse distance, cut-off 69 km\",tw,lgrdp,direct,-5.541342308192129,-8.7475404707428,-2.3731422298828178\n\"Inverse distance, cut-off 69 km\",tw,tpt,direct,-0.04105780774858734,-0.09483864230306821,0.01086684430957557\n\"Inverse distance, cut-off 69 km\",tw,sanit,direct,-0.03698982718620405,-0.052487334229939654,-0.02064047821982535\n\"Inverse distance, cut-off 69 km\",tw,manuf,indirect,0.010379751011031528,-0.07324100440690792,0.0935362207560968\n\"Inverse distance, cut-off 69 km\",tw,lgrdp,indirect,3.1161220660145017,-2.14921356548911,9.02024162685848\n\"Inverse distance, cut-off 69 km\",tw,tpt,indirect,-0.11501736935930046,-0.2660624387216603,0.020794872121705065\n\"Inverse distance, cut-off 69 km\",tw,sanit,indirect,-0.08019223187900232,-0.12491093710045509,-0.037777433512122885\n\"Inverse distance, cut-off 69 km\",tw,manuf,total,0.02214679570598504,-0.06807848806807638,0.1112376394131038\n\"Inverse distance, cut-off 69 km\",tw,lgrdp,total,-2.425220242177627,-8.239674600244957,4.2466407070394405\n\"Inverse distance, cut-off 69 km\",tw,tpt,total,-0.1560751771078878,-0.31489057880179,-0.010905552091392986\n\"Inverse distance, cut-off 69 km\",tw,sanit,total,-0.11718205906520637,-0.1651513712966106,-0.07258165908459223\n", "E2": "set,FE,W,AIC,delta_AIC,elpd_waic,se_waic,p_waic,diff_waic,se_diff_waic,elpd_loo,se_loo,diff_loo,se_diff_loo,share_pareto_k_gt_0_7_before_refit,max_pareto_k\ncore,ind,Queen contiguity,20.49265904208678,0.7542714136605753,-28.906188247347913,10.920100519180753,42.184075409614735,-5.418863391570884,5.554586190550712,-31.271880703749638,11.146465529954357,-5.589270827302342,5.696108074158778,0.005714285714285714,0.7204512776901002\ncore,ind,4-nearest neighbours (great circle),19.738387628426203,0.0,-23.48732485577703,9.246394374335642,40.07642230544214,0.0,0.0,-25.682609876447295,9.431700297641836,0.0,0.0,0.0,0.6775853821103862\ncore,ind,Economic k=4 (2020 profile),27.09670106883631,7.358313440410107,-29.910236595791908,10.298845323238115,40.94476835094212,-6.422911740014882,7.216213545772666,-32.44346848721146,10.577552236465232,-6.760858610764166,7.44886934584372,0.011428571428571429,0.8172376825120704\ncore,tw,Queen contiguity,-24.84912201366315,0.0,-4.379389740885536,10.87908308226052,44.20988244068432,0.0,0.0,-7.706942208021573,11.30004294692904,0.0,0.0,0.05142857142857143,1.0081592926109253\ncore,tw,4-nearest neighbours (great circle),-18.32234983837865,6.526772175284499,-7.749480743428654,9.479463955459176,42.70468911836636,-3.370091002543114,5.6007050272562235,-10.818185591151577,9.827633491049367,-3.111243383130004,5.832542883497869,0.04,1.0536788284059797\ncore,tw,Economic k=4 (2020 profile),-1.151783534461707,23.69733847920144,-20.112047366685225,9.790779941583718,42.812212730180796,-15.732657625799689,9.408101701486531,-23.42327241082234,10.184979351521305,-15.716330202800767,9.818445605224618,0.04,0.9473111442145815\nextended,ind,Queen contiguity,20.49265904208678,16.15856640450065,-28.906188247347913,10.920100519180753,42.184075409614735,-16.58383624269829,5.214151516332623,-31.271880703749638,11.146465529954357,-17.028547025706068,5.3769917539411365,0.005714285714285714,0.7204512776901002\nextended,ind,4-nearest neighbours (great circle),19.738387628426203,15.404294990840075,-23.48732485577703,9.246394374335642,40.07642230544214,-11.164972851127402,5.509686740872328,-25.682609876447295,9.431700297641836,-11.439276198403727,5.638420920277108,0.0,0.6775853821103862\nextended,ind,Economic k=4 (2020 profile),27.09670106883631,22.762608431250182,-29.910236595791908,10.298845323238115,40.94476835094212,-17.587884591142288,7.177431298434378,-32.44346848721146,10.577552236465232,-18.200134809167892,7.36490133138113,0.011428571428571429,0.8172376825120704\nextended,ind,Inverse distance (non-truncated),4.334092637586128,0.0,-12.322352004649623,9.621237138583915,37.22316916756718,0.0,0.0,-14.24333367804357,9.80820326563151,0.0,0.0,0.005714285714285714,0.8669863297375981\nextended,ind,\"Inverse distance, cut-off 69 km\",21.117679944861166,16.783587307275038,-25.65361210739766,9.813119588554324,39.45696937029554,-13.331260102748038,3.6459967871650276,-27.707627744922036,10.010210783586663,-13.464294066878466,3.6872681885220646,0.0,0.6939853796444352\nextended,tw,Queen contiguity,-24.84912201366315,0.0,-4.379389740885536,10.87908308226052,44.20988244068432,-2.0168465808661136,3.63020998454989,-7.706942208021573,11.30004294692904,-1.7423123208855245,3.7222681160744915,0.05142857142857143,1.0081592926109253\nextended,tw,4-nearest neighbours (great circle),-18.32234983837865,6.526772175284499,-7.749480743428654,9.479463955459176,42.70468911836636,-5.38693758340923,5.262901735998113,-10.818185591151577,9.827633491049367,-4.853555704015529,5.46049441665035,0.04,1.0536788284059797\nextended,tw,Economic k=4 (2020 profile),-1.151783534461707,23.69733847920144,-20.112047366685225,9.790779941583718,42.812212730180796,-17.749504206665797,8.56666374883344,-23.42327241082234,10.184979351521305,-17.458642523686294,8.959219209500395,0.04,0.9473111442145815\nextended,tw,Inverse distance (non-truncated),-24.458133378893976,0.39098863476917245,-2.362543160019424,10.0097854464893,46.23752917361465,0.0,0.0,-5.964629887136049,10.397248011894135,0.0,0.0,0.04,0.9781973603750461\nextended,tw,\"Inverse distance, cut-off 69 km\",-21.76942739155885,3.0796946221042987,-7.420093392438506,10.165255941944887,44.844600326459684,-5.05755023241908,3.952135746729174,-10.849254117547222,10.535739221949818,-4.884624230411174,4.108462848443135,0.05714285714285714,0.9338490091907398\n", "E4": "set,FE,W,akaike,stacking_loo,bma_P_W_given_SDM,bma_P_W_marginal_SDM_SLX,bma_P_SDM_given_W\ncore,ind,Queen contiguity,0.40081592146447825,0.26563688667288643,0.3516709053088958,0.35167441070609906,0.9999844497914984\ncore,ind,4-nearest neighbours (great circle),0.5844305650058681,0.37707857361389235,0.6299848135455132,0.6299814072478568,0.9999998244215512\ncore,ind,Economic k=4 (2020 profile),0.014753513529653633,0.3572845397132212,0.018344281145590974,0.018344182046044162,0.9999998196733941\ncore,tw,Queen contiguity,0.96314452408747,0.7510735823112485,0.9583167204350251,0.9378755033418749,0.2147851945619005\ncore,tw,4-nearest neighbours (great circle),0.03684859129543089,0.0494800252367692,0.04167362653237626,0.06211324228383479,0.1410319672748222\ncore,tw,Economic k=4 (2020 profile),6.88461709911809e-06,0.19944639245198217,9.653032598629819e-06,1.1254374290313211e-05,0.18029466947106274\nextended,ind,Queen contiguity,0.0003095835413754617,1.4987929852625984e-13,0.00019062477297573948,0.00019062773669049077,0.9999844497914985\nextended,ind,4-nearest neighbours (great circle),0.0004514044335901296,0.04495324269629504,0.00034148606053945073,0.0003414861194467527,0.9999998244215512\nextended,ind,Economic k=4 (2020 profile),1.1395368102027214e-05,0.10278581594618569,9.943598904520926e-06,9.943600667033375e-06,0.9999998196733942\nextended,ind,Inverse distance (non-truncated),0.999001122866784,0.8522609400265236,0.9992590166573635,0.9992590135977979,0.9999999999861564\nextended,ind,\"Inverse distance, cut-off 69 km\",0.00022649379014826914,1.3308458464523956e-09,0.0001989289102167731,0.0001989289453977244,0.999999820072476\nextended,tw,Queen contiguity,0.4819027136628308,0.40899476753687125,0.2765355670412367,0.44267847421535333,0.21478519456190048\nextended,tw,4-nearest neighbours (great circle),0.018436938274393418,5.913528004442835e-09,0.012025502318861766,0.02931753225753421,0.1410319672748222\nextended,tw,Economic k=4 (2020 profile),3.444670638332185e-06,0.16522903039741876,2.7855162979081387e-06,5.312079504509976e-06,0.18029466947106276\nextended,tw,Inverse distance (non-truncated),0.39633029322947216,0.4257689764991812,0.6281265441829966,0.42434402664768134,0.5089449423271724\nextended,tw,\"Inverse distance, cut-off 69 km\",0.10332661016266537,7.2196530007887065e-06,0.08330960094060696,0.10365465479992675,0.27634269824456786\n", "T01": "W,directed,mean_degree,min_in_degree,max_in_degree,density,edge_reciprocity,weight_asymmetry_frob,min_real_eig,n_complex_eig,max_mod_eig,rho_lower_bound,weak_components,strong_components,clustering,diameter\nQueen contiguity,False,4.228571428571429,1.0,8.0,0.12436974789915967,1.0,0.7301902739089305,-0.6021790829714877,0,0.9999999999999989,-0.9899,1,1,0.411371237458194,9\n4-nearest neighbours (great circle),True,4.0,1.0,7.0,0.11764705882352941,0.8,0.6324555320336759,-0.5983729384980256,16,0.9999999999999996,-0.9899,1,1,0.5045045045045045,10\nEconomic k=4 (2020 profile),True,4.0,0.0,9.0,0.11764705882352941,0.6142857142857143,0.8783100656536799,-0.48641386811216947,10,1.0000000000000047,-0.9899,1,5,0.5073068893528184,7\n", "E1b": "W,FE,elpd_psis,elpd_psis_plus_exact,n_refit,elpd_waic\nQueen contiguity,ind,-31.122341784789334,-31.271880703749638,1,-28.906188247347913\nQueen contiguity,tw,-7.904339401906695,-7.706942208021573,9,-4.379389740885536\n4-nearest neighbours (great circle),ind,-25.682609876447295,-25.682609876447295,0,-23.48732485577703\n4-nearest neighbours (great circle),tw,-10.899709288370804,-10.818185591151577,7,-7.749480743428654\nEconomic k=4 (2020 profile),ind,-32.31747516103273,-32.44346848721146,2,-29.910236595791908\nEconomic k=4 (2020 profile),tw,-23.586182851002356,-23.42327241082234,7,-20.112047366685225\nInverse distance (non-truncated),ind,-14.29203384188752,-14.24333367804357,1,-12.322352004649623\nInverse distance (non-truncated),tw,-6.14236238058623,-5.964629887136049,7,-2.362543160019424\n\"Inverse distance, cut-off 69 km\",ind,-27.707627744922036,-27.707627744922036,0,-25.65361210739766\n\"Inverse distance, cut-off 69 km\",tw,-11.204373844143127,-10.849254117547222,10,-7.420093392438506\n"}
REF_MS = {  # angka di naskah (v8)
    "rho_queen_ind": 0.425, "rho_queen_tw": -0.135, "rho_queen_ind_sub": -0.184, "rho_queen_tw_sub": -0.145,
    "se_rho_queen_ind": 0.078, "se_rho_queen_tw": 0.119,
    "share_pov_unit_pct": 2.2, "share_pov_tw_pct": 0.44, "sd_within_pov": 0.503, "wy_year_R2_pct": 92,
    "manuf_total_queen_ind": 0.049, "manuf_direct_queen_ind": 0.019, "manuf_indirect_queen_ind": 0.031,
    "dAIC_tw_queen_knn": 6.5, "dAIC_tw_queen_econ": 23.7,
    "tau_queen_ind_T7": 0.537, "tau_queen_tw_T7": 0.572,
    "T8_SR_ind": -0.113, "T8_SR_tw": -0.085, "T8_LR_ind": -0.127, "T8_LR_tw": -0.169,
    "S30_tau_fe_ind": 0.377, "S30_tau_fe_tw": 0.407, "S30_tau_re": 0.974, "S30_tau_pooled": 0.975,
    "S5_rho_unit_mean": 0.453, "S5_rho_tw_mean": -0.154,
    "S32_AIC_queen": 0.97, "S32_AIC_knn4": 0.92, "S32_AIC_econ": 0.997,
}
print("DATA_PATH     :", DATA_PATH)
print("PRESAMPLE_PATH:", PRESAMPLE_PATH)
print("GEO_PATH      :", GEO_PATH)
assert DATA_PATH, "File data tidak ditemukan: isi DATA_PATH manual"

In [ ]:
# Sel 2 — Import, deteksi kolom, dan data
import numpy as np, pandas as pd, time, warnings
warnings.filterwarnings('ignore')
from scipy.optimize import minimize_scalar
from scipy.stats import chi2
from joblib import Parallel, delayed

if QUICK:
    N_DRAWS, N_MIX, BAYES_S, REFIT_ITER, REFIT_BURN = 500, 2000, 400, 300, 100
    B_EMP, MAX_ITER, BAYES_ITER_EMP, BAYES_BURN_EMP = 19, 5, 1200, 400
    MC_RHO_REPS, MC_W_REPS = 6, 6

df = pd.read_csv(DATA_PATH)
print("Kolom data:", list(df.columns))
W_codes = set(pd.read_csv(W_PATHS["queen"], index_col=0).index.astype(str))

def _norm(v):
    try:
        return str(int(float(v)))
    except Exception:
        return str(v).strip()

def _pick(aliases, exclude=()):
    cols = [c for c in df.columns if c not in exclude]
    low = {c.lower().strip(): c for c in cols}
    for a in aliases:                      # cocok persis
        if a in low:
            return low[a]
    for a in aliases:                      # cocok sebagian
        for lc, c in low.items():
            if a in lc:
                return c
    return None

# ID: (1) kolom berisi kode BPS 3301 ... 3376, atau (2) kolom nama kab/kota -> kode BPS
BPS_JATENG = {
    "cilacap": "3301", "banyumas": "3302", "purbalingga": "3303", "banjarnegara": "3304",
    "kebumen": "3305", "purworejo": "3306", "wonosobo": "3307", "magelang": "3308",
    "boyolali": "3309", "klaten": "3310", "sukoharjo": "3311", "wonogiri": "3312",
    "karanganyar": "3313", "sragen": "3314", "grobogan": "3315", "blora": "3316",
    "rembang": "3317", "pati": "3318", "kudus": "3319", "jepara": "3320", "demak": "3321",
    "semarang": "3322", "temanggung": "3323", "kendal": "3324", "batang": "3325",
    "pekalongan": "3326", "pemalang": "3327", "tegal": "3328", "brebes": "3329",
    "kota magelang": "3371", "kota surakarta": "3372", "kota salatiga": "3373",
    "kota semarang": "3374", "kota pekalongan": "3375", "kota tegal": "3376",
}
def _name_to_code(v):
    t = str(v).lower()
    for ch in ".,_-()":
        t = t.replace(ch, " ")
    t = " ".join(t.split())
    is_kota = t.startswith("kota ") or t.endswith(" kota") or " kota " in t
    for w in ["kabupaten", "kab", "kota"]:
        t = " ".join(x for x in t.split() if x != w)
    t = t.replace("solo", "").strip()
    if t in ("surakarta", "salatiga"):
        is_kota = True
    return BPS_JATENG.get(("kota " + t) if is_kota else t)

if COL_ID is None:
    for c in df.columns:
        if set(df[c].map(_norm).unique()) >= W_codes:
            COL_ID = c; break
if COL_ID is None:
    for c in df.columns:
        if not pd.api.types.is_numeric_dtype(df[c]):
            codes = df[c].map(_name_to_code)
            if set(codes.dropna().unique()) >= W_codes:
                bad = sorted(df.loc[codes.isna(), c].astype(str).unique())
                if bad:
                    print(f"Nama tak dikenal (baris diabaikan): {bad}")
                df["kode__"] = codes
                print(f"ID dibuat dari nama di kolom '{c}' -> kode BPS")
                COL_ID = "kode__"; break
if COL_YEAR is None:
    COL_YEAR = _pick(["tahun", "year", "thn", "periode"])
if COL_Y is None:
    COL_Y = _pick(["poverty", "kemiskinan", "persen_miskin", "pct_miskin", "p0", "miskin"])
ALIASES = {
    "manuf": ["pertumbuhanindustri", "pertumbuhan_industri", "manuf", "growth_manuf",
              "pertumbuhan", "manufaktur", "industry_growth"],
    "lgrdp": ["lgrdp", "log_grdp", "ln_grdp", "log_pdrb", "ln_pdrb", "lpdrb", "lnpdrb", "logpdrb"],
    "tpt":   ["tpt", "unemp", "pengangguran", "unemployment"],
    "sanit": ["sanit", "sanitasi", "sanitation", "jamban"],
}
if COL_HDI is None:
    COL_HDI = _pick(["ipm", "hdi"], exclude=[COL_ID, COL_YEAR, COL_Y])
if COL_TPAK is None:
    COL_TPAK = _pick(["tpak", "lfpr", "partisipasi"], exclude=[COL_ID, COL_YEAR, COL_Y])
for k in COL_X:
    if COL_X[k] is None:
        COL_X[k] = _pick(ALIASES[k], exclude=[COL_ID, COL_YEAR, COL_Y])
# log GRDP: jika tidak ada kolom log, ambil log dari kolom PDRB/GRDP level
if COL_X["lgrdp"] is None:
    lvl = _pick(["pdrb_adhk", "pdrb", "grdp"], exclude=[COL_ID, COL_YEAR, COL_Y])
    if lvl is not None:
        v = df[lvl].astype(float)
        if v.median() > 30:
            df["lgrdp__"] = np.log(v); print(f"log GRDP = log(kolom '{lvl}')")
        else:
            df["lgrdp__"] = v; print(f"kolom '{lvl}' sudah berskala log, dipakai apa adanya")
        COL_X["lgrdp"] = "lgrdp__"

print("\nPemetaan kolom (CEK!):")
print(f"  id={COL_ID} | tahun={COL_YEAR} | y={COL_Y}")
for k, v in COL_X.items():
    print(f"  {k:5s} -> {v}")
miss = [n for n, v in [("COL_ID", COL_ID), ("COL_YEAR", COL_YEAR), ("COL_Y", COL_Y)] if v is None] \
       + [f"COL_X['{k}']" for k, v in COL_X.items() if v is None]
assert not miss, f"Tidak terdeteksi: {miss} -> isi manual di Sel 1"

df = df[df[COL_ID].notna()].copy()
df[COL_ID] = df[COL_ID].map(_norm)
df[COL_YEAR] = df[COL_YEAR].astype(int)
df = df[df[COL_YEAR].isin(YEARS) & df[COL_ID].isin(W_codes)].sort_values([COL_YEAR, COL_ID])
XCOLS = list(COL_X.values())
ids = sorted(df[COL_ID].unique())
N, T_ALL, K = len(ids), len(YEARS), len(XCOLS)
assert len(df) == N * T_ALL, f"panel harus balanced: {len(df)} baris, N={N}, T={T_ALL}"
assert not df[[COL_Y] + XCOLS].isna().any().any(), "ada nilai kosong di y/X"
Y = df.pivot(index=COL_YEAR, columns=COL_ID, values=COL_Y).loc[YEARS, ids].values.astype(float)
X = np.stack([df.pivot(index=COL_YEAR, columns=COL_ID, values=c).loc[YEARS, ids].values.astype(float)
              for c in XCOLS], axis=2)
COL_X = list(COL_X)        # nama pendek untuk label output
piv = lambda c: df.pivot(index=COL_YEAR, columns=COL_ID, values=c).loc[YEARS, ids].values.astype(float)
HDI = piv(COL_HDI) if COL_HDI else None
TPAK = piv(COL_TPAK) if COL_TPAK else None
print(f"  hdi   -> {COL_HDI}\n  tpak  -> {COL_TPAK}")

def load_W(path):
    w = pd.read_csv(path, index_col=0)
    w.index, w.columns = w.index.astype(str), w.columns.astype(str)
    w = w.loc[ids, ids].values.astype(float)
    assert np.allclose(w.sum(1), 1), "W harus row-standardised"
    return w

WS = {k: load_W(p) for k, p in W_PATHS.items()}
EVS = {k: np.linalg.eigvals(w) for k, w in WS.items()}
if COL_ID == "kode__":
    _m = df[[c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c]) and c != COL_ID][0]].groupby(df[COL_ID]).first()
    print("\nNama -> kode BPS:", dict(_m))
print("\nN =", N, "T =", T_ALL, "| W:", list(WS))
print("Rata-rata per tahun (cek skala):")
print(df.groupby(COL_YEAR)[[COL_Y] + XCOLS].mean().round(3).to_string())

In [ ]:
# Sel 3 — Estimator statis: SDM, SAR, SLX, SEM, SDEM (QML, Lee & Yu 2010a), kovarians, impacts
from scipy.optimize import minimize_scalar, minimize
from scipy.stats import chi2, norm, spearmanr
from scipy.special import logsumexp
from joblib import Parallel, delayed

LO, HI = -0.99, 0.999
def lagW(W, A):
    return np.einsum("ij,tj...->ti...", W, A)

def demean(A, fe):
    A = A - A.mean(0, keepdims=True)
    if fe == "tw":
        A = A - A.mean(1, keepdims=True)
    return A.reshape(A.shape[0] * A.shape[1], -1)

def neff_of(T, n, fe):
    return (n - 1) * (T - 1) if fe == "tw" else n * (T - 1)

def ldet(r, ev, T, fe):
    ld = np.sum(np.log(np.abs(1 - r * ev))).real
    if fe == "tw":
        ld -= np.log(1 - r)
    return (T - 1) * ld

def fit_lag(Yt, Z, W, ev, fe):
    T, n = Yt.shape
    zt = demean(Z, fe); yt = demean(Yt, fe)[:, 0]; wyt = demean(lagW(W, Yt), fe)[:, 0]
    neff = neff_of(T, n, fe)
    P = np.linalg.solve(zt.T @ zt, zt.T)
    b0, b1 = P @ yt, P @ wyt
    e0, e1 = yt - zt @ b0, wyt - zt @ b1
    a, bb, c = e0 @ e0, 2 * e0 @ e1, e1 @ e1
    def nll(d):
        s2 = (a - d * bb + d * d * c) / neff
        return 0.5 * neff * (np.log(2 * np.pi * s2) + 1) - ldet(d, ev, T, fe)
    r = minimize_scalar(nll, bounds=(LO, HI), method="bounded", options={"xatol": 1e-8})
    d = r.x
    return dict(d=d, b=b0 - d * b1, s2=(a - d * bb + d * d * c) / neff, ll=-r.fun, neff=neff)

def fit_err(Yt, Z, W, ev, fe):
    T, n = Yt.shape
    neff = neff_of(T, n, fe)
    WY, WZ = lagW(W, Yt), lagW(W, Z)
    def ols(l):
        ys = demean(Yt - l * WY, fe)[:, 0]; zs = demean(Z - l * WZ, fe)
        b = np.linalg.lstsq(zs, ys, rcond=None)[0]; e = ys - zs @ b
        return b, e @ e / neff
    def nll(l):
        _, s2 = ols(l)
        return 0.5 * neff * (np.log(2 * np.pi * s2) + 1) - ldet(l, ev, T, fe)
    r = minimize_scalar(nll, bounds=(LO, HI), method="bounded", options={"xatol": 1e-8})
    b, s2 = ols(r.x)
    return dict(d=r.x, b=b, s2=s2, ll=-r.fun, neff=neff)

def fit_ols(Yt, Z, fe):
    T, n = Yt.shape
    neff = neff_of(T, n, fe)
    zt = demean(Z, fe); yt = demean(Yt, fe)[:, 0]
    b = np.linalg.lstsq(zt, yt, rcond=None)[0]; e = yt - zt @ b; s2 = e @ e / neff
    ll = -0.5 * neff * (np.log(2 * np.pi * s2) + 1)
    return dict(d=0.0, b=b, s2=s2, ll=ll, neff=neff)

def full_ll(theta, Yt, Z, W, ev, fe, kind):
    """theta = [spatial, b..., log s2]; kind in lag / err / ols."""
    T, n = Yt.shape
    neff = neff_of(T, n, fe)
    s2 = np.exp(theta[-1]); b = theta[1:-1]; r = theta[0]
    if kind == "lag":
        e = demean(Yt - r * lagW(W, Yt), fe)[:, 0] - demean(Z, fe) @ b
    elif kind == "err":
        e = demean(Yt - r * lagW(W, Yt), fe)[:, 0] - demean(Z - r * lagW(W, Z), fe) @ b
    else:
        e = demean(Yt, fe)[:, 0] - demean(Z, fe) @ b
    ld = ldet(r, ev, T, fe) if kind != "ols" else 0.0
    return -0.5 * neff * np.log(2 * np.pi * s2) + ld - e @ e / (2 * s2)

def num_hess(f, x, h=1e-4):
    k = len(x); H = np.zeros((k, k))
    for i in range(k):
        for j in range(i, k):
            ei = np.zeros(k); ej = np.zeros(k); ei[i] = h; ej[j] = h
            H[i, j] = H[j, i] = (f(x + ei + ej) - f(x + ei - ej) - f(x - ei + ej) + f(x - ei - ej)) / (4 * h * h)
    return H

def estimate(Yt, Xc, W, ev, fe, model):
    """model: SDM, SAR, SLX, SEM, SDEM. Xc (T,N,K). Returns dict with coefficients, cov, ll, aic."""
    WX = lagW(W, Xc)
    Z = np.concatenate([Xc, WX], axis=2) if model in ("SDM", "SLX", "SDEM") else Xc
    kind = {"SDM": "lag", "SAR": "lag", "SLX": "ols", "SEM": "err", "SDEM": "err"}[model]
    f = {"lag": fit_lag, "err": fit_err}.get(kind)
    r = f(Yt, Z, W, ev, fe) if f else fit_ols(Yt, Z, fe)
    th = np.r_[r["d"], r["b"], np.log(r["s2"])]
    ll_f = lambda t: full_ll(t, Yt, Z, W, ev, fe, kind)
    if kind == "ols":
        H = num_hess(lambda t: ll_f(np.r_[0.0, t]), th[1:])
        cov = np.zeros((len(th), len(th))); cov[1:, 1:] = np.linalg.pinv(-H)
    else:
        cov = np.linalg.pinv(-num_hess(ll_f, th))
    kpar = len(r["b"]) + 1 + (kind != "ols")
    vars_ = COL_X + ["hdi"] if Xc.shape[2] == len(COL_X) + 1 else COL_X
    return dict(model=model, fe=fe, kind=kind, theta=th, cov=cov, ll=r["ll"], k=kpar,
                aic=-2 * r["ll"] + 2 * kpar, Z=Z, K=Xc.shape[2], s2=r["s2"], vars=vars_)

def impacts_draws(fit, ev, ndraw, rng, point=False):
    """Direct/indirect/total for each X: array (ndraw, K, 3)."""
    K = fit["K"]
    th = fit["theta"][None, :] if point else rng.multivariate_normal(fit["theta"], fit["cov"], ndraw,
                                                                      method="eigh")
    r = th[:, 0][:, None]; b = th[:, 1:1 + K]
    t = th[:, 1 + K:1 + 2 * K] if fit["model"] in ("SDM", "SLX", "SDEM") else np.zeros_like(b)
    if fit["model"] in ("SDM", "SAR"):
        lam = ev[None, None, :]
        f = (b[..., None] + t[..., None] * lam) / (1 - r[..., None] * lam)
        direct = f.mean(-1).real
        total = (b + t) / (1 - r)
    else:                                           # SLX, SDEM: no feedback; SEM: no spillover
        direct, total = b, b + t
    return np.stack([direct, total - direct, total], axis=-1)

def impact_table(fit, ev, rng, label):
    pt = impacts_draws(fit, ev, 1, rng, point=True)[0]
    dr = impacts_draws(fit, ev, N_DRAWS, rng)
    lo, hi = np.quantile(dr, [.025, .975], axis=0)
    rows = []
    for j, v in enumerate(fit["vars"]):
        for e, en in enumerate(["direct", "indirect", "total"]):
            rows.append(dict(spec=label, variable=v, effect=en, estimate=pt[j, e], lwr95=lo[j, e], upr95=hi[j, e]))
    return rows, dr

def coef_table(fit, label):
    K = fit["K"]; nm = ["rho" if fit["kind"] == "lag" else ("lambda" if fit["kind"] == "err" else "-")]
    nm += [f"b_{v}" for v in fit["vars"]] + ([f"th_{v}" for v in fit["vars"]] if fit["model"] in ("SDM", "SLX", "SDEM") else [])
    nm += ["log_s2"]
    se = np.sqrt(np.clip(np.diag(fit["cov"]), 0, None))
    out = [dict(spec=label, param=n, est=e, se=s, z=e / s if s > 0 else np.nan)
           for n, e, s in zip(nm, fit["theta"], se) if n != "-"]
    out.append(dict(spec=label, param="sigma2", est=fit["s2"], se=np.nan, z=np.nan))
    out.append(dict(spec=label, param="logLik", est=fit["ll"], se=np.nan, z=np.nan))
    out.append(dict(spec=label, param="AIC", est=fit["aic"], se=np.nan, z=np.nan))
    return out

def save(df_, name):
    df_ = pd.DataFrame(df_)
    df_.to_csv(os.path.join(RES, name + ".csv"), index=False)
    return df_

rng = np.random.default_rng(SEED)
print("Estimator siap.")

In [ ]:
# Sel 4 — Matriks W: matriks beku, rekonstruksi ekonomi/KNN, inverse distance, varian ekonomi, pra-sampel
def rowstd(A):
    A = np.asarray(A, float).copy(); np.fill_diagonal(A, 0)
    s = A.sum(1, keepdims=True); s[s == 0] = 1
    return A / s

def knn_W(D, k):
    A = np.zeros_like(D)
    for i in range(len(D)):
        d = D[i].copy(); d[i] = np.inf
        A[i, np.argsort(d, kind="stable")[:k]] = 1
    return rowstd(A)

def attr_dist(F, metric):
    Fz = (F - F.mean(0)) / F.std(0, ddof=1)
    diff = Fz[:, None, :] - Fz[None, :, :]
    return np.abs(diff).sum(-1) if metric == "manhattan" else np.sqrt((diff ** 2).sum(-1))

def edges(W):
    return set(zip(*np.nonzero(W > 0)))

def overlap(Wa, Wb):
    a, b = edges(Wa), edges(Wb)
    return len(a & b) / len(a | b)

ALLW = {"queen": WS["queen"], "knn4": WS["knn4"], "econ": WS["econ"]}
WLAB = {"queen": "Queen contiguity", "knn4": "4-nearest neighbours (great circle)",
        "econ": "Economic k=4 (2020 profile)", "idw_full": "Inverse distance (non-truncated)",
        "idw69": "Inverse distance, cut-off 69 km"}
w_checks = []

# --- ekonomi 2020: log GRDP, TPT, TPAK (standardised, Euclidean, k = 4) ---
i20 = YEARS.index(2020)
prof20 = None
if TPAK is not None:
    prof20 = np.c_[X[i20, :, COL_X.index("lgrdp")], X[i20, :, COL_X.index("tpt")], TPAK[i20]]
    W_e = knn_W(attr_dist(prof20, "euclid"), 4)
    w_checks.append(dict(check="econ k=4 2020 rebuilt vs frozen (edge overlap)", value=overlap(W_e, WS["econ"])))
ECON_VAR = {}
if prof20 is not None:
    for k in [3, 4, 5, 6]:
        for m in ["euclid", "manhattan"]:
            ECON_VAR[f"econ_k{k}_{m}"] = knn_W(attr_dist(prof20, m), k)
    ECON_VAR["econ_noregressors_k4"] = knn_W(attr_dist(TPAK[i20][:, None], "euclid"), 4)

# --- pra-sampel 2019 ---
PRE = {}
if PRESAMPLE_PATH:
    pre = pd.read_excel(PRESAMPLE_PATH) if PRESAMPLE_PATH.endswith("xlsx") else pd.read_csv(PRESAMPLE_PATH)
    print("Kolom pra-sampel:", list(pre.columns))
    idc = None
    for c in pre.columns:
        if set(pre[c].map(_norm).unique()) >= W_codes:
            idc = c; pre["kode__"] = pre[c].map(_norm); break
    if idc is None:
        for c in pre.columns:
            if not pd.api.types.is_numeric_dtype(pre[c]):
                codes = pre[c].map(_name_to_code)
                if set(codes.dropna().unique()) >= W_codes:
                    pre["kode__"] = codes; break
    yc = next((c for c in pre.columns if c.lower() in ("tahun", "year")), None)
    if yc is not None:
        pre = pre[pre[yc] == 2019]
    pre = pre[pre["kode__"].isin(ids)].drop_duplicates("kode__").set_index("kode__").loc[ids]
    lowc = {c.lower(): c for c in pre.columns}
    pick = lambda al: next((lowc[a] for a in lowc for x in al if x in a), None)
    c_g, c_u, c_l = pick(["pdrb", "grdp"]), pick(["tpt", "unemp", "pengangguran"]), pick(["tpak", "lfpr"])
    print(f"pra-sampel: PDRB={c_g}, TPT={c_u}, TPAK={c_l}")
    if c_g and c_u and c_l:
        g = pre[c_g].astype(float).values; g = np.log(g) if np.median(g) > 30 else g
        F19 = np.c_[g, pre[c_u].astype(float).values, pre[c_l].astype(float).values]
        PRE["pre2019_k4"] = knn_W(attr_dist(F19, "euclid"), 4)
        PRE["pre2019_labour_k4"] = knn_W(attr_dist(F19[:, 1:], "euclid"), 4)
else:
    print("Data pra-sampel 2019 tidak ditemukan: Table S2b/S22/S23a dilewati.")

# --- geo: centroid -> KNN-4 check, inverse distance ---
LONLAT = None
if GEO_PATH:
    try:
        if GEO_PATH.endswith(".csv"):
            g = pd.read_csv(GEO_PATH)
        else:
            import geopandas as gpd
            g = gpd.read_file(GEO_PATH)
            g = g.to_crs(4326) if g.crs is not None else g
            cen = g.geometry.centroid
            g = pd.DataFrame(g.drop(columns="geometry")); g["lon__"], g["lat__"] = cen.x.values, cen.y.values
        key = None
        for c in g.columns:
            if c in ("lon__", "lat__"):
                continue
            v = g[c].astype(str).str.replace(".", "", regex=False).map(_norm)
            if set(v.unique()) >= W_codes:
                g["kode__"] = v; key = c; break
        if key is None:
            for c in g.columns:
                if not pd.api.types.is_numeric_dtype(g[c]):
                    codes = g[c].map(_name_to_code)
                    if set(codes.dropna().unique()) >= W_codes:
                        g["kode__"] = codes; key = c; break
        if key is None:                       # shapefile tanpa prefiks Kota: kode >= 71 = kota
            raise ValueError("kolom kode/nama di file geo tidak cocok dengan 35 kab/kota")
        lowc = {c.lower(): c for c in g.columns}
        lonc = "lon__" if "lon__" in g else next(lowc[a] for a in lowc if a in ("lon", "longitude", "x", "lng", "bujur"))
        latc = "lat__" if "lat__" in g else next(lowc[a] for a in lowc if a in ("lat", "latitude", "y", "lintang"))
        g = g.drop_duplicates("kode__").set_index("kode__").loc[ids]
        LONLAT = g[[lonc, latc]].astype(float).values
        print(f"Geo: kunci '{key}', koordinat ({lonc}, {latc})")
    except Exception as e:
        print("Geo tidak terbaca:", e)
if LONLAT is not None:
    lo, la = np.radians(LONLAT[:, 0]), np.radians(LONLAT[:, 1])
    dlo, dla = lo[:, None] - lo[None, :], la[:, None] - la[None, :]
    a = np.sin(dla / 2) ** 2 + np.cos(la[:, None]) * np.cos(la[None, :]) * np.sin(dlo / 2) ** 2
    DKM = 2 * 6371.0 * np.arcsin(np.sqrt(a))
    w_checks.append(dict(check="KNN-4 rebuilt from centroids vs frozen (edge overlap)", value=overlap(knn_W(DKM, 4), WS["knn4"])))
    nn = np.where(np.eye(N, dtype=bool), np.inf, DKM).min(1)
    CUT = 2 * nn.max()
    w_checks.append(dict(check="cut-off = 2 x largest NN distance (km; naskah 69)", value=CUT))
    inv = np.where(np.eye(N, dtype=bool), 0, 1 / np.where(DKM == 0, np.inf, DKM))
    ALLW["idw_full"] = rowstd(inv)
    ALLW["idw69"] = rowstd(np.where(DKM <= 69.0, inv, 0))
else:
    print("Koordinat tidak tersedia: matriks inverse distance (Table S15, set diperluas) dilewati.")

EV = {k: np.linalg.eigvals(w) for k, w in {**ALLW, **ECON_VAR, **PRE}.items()}
CORE = ["queen", "knn4", "econ"]
EXT = CORE + [k for k in ["idw_full", "idw69"] if k in ALLW]
print(pd.DataFrame(w_checks).to_string(index=False))
save(w_checks, "W_checks")

In [ ]:
# Sel 5 — Table 1, S1a, S1b, S2a, S2b: grafik dan spektrum
import networkx as nx
from scipy.sparse.csgraph import connected_components

def graph_row(name, W):
    A = (W > 0).astype(int); ev = np.linalg.eigvals(W)
    E = A.sum(); rec = (A * A.T).sum() / E
    G = nx.from_numpy_array(np.maximum(A, A.T))
    real = ev.real[np.abs(ev.imag) < 1e-6]
    return dict(W=WLAB.get(name, name), directed=bool((A != A.T).any()), mean_degree=A.sum(1).mean(),
                min_in_degree=A.sum(0).min(), max_in_degree=A.sum(0).max(), density=E / (N * (N - 1)),
                edge_reciprocity=rec, weight_asymmetry_frob=np.linalg.norm(W - W.T) / np.linalg.norm(W),
                min_real_eig=real.min(), n_complex_eig=int((np.abs(ev.imag) > 1e-6).sum()),
                max_mod_eig=np.abs(ev).max(), rho_lower_bound=max(LO, 1 / real.min()),
                weak_components=connected_components(A, directed=True, connection="weak")[0],
                strong_components=connected_components(A, directed=True, connection="strong")[0],
                isolated_in=int((A.sum(0) == 0).sum()),
                clustering=nx.transitivity(G),
                diameter=(nx.diameter(nx.from_numpy_array(A, create_using=nx.DiGraph))
                          if connected_components(A, directed=True, connection="strong")[0] == 1 else nx.diameter(G)))

T1 = save([graph_row(k, ALLW[k]) for k in EXT], "T1_S1ab_graph_spectrum")
print(T1.round(3).to_string(index=False))
if ECON_VAR or PRE:
    allv = {"econ": WS["econ"], **ECON_VAR, **PRE}
    names = list(allv)
    S2 = save([dict(W_a=a, W_b=b, edge_jaccard=overlap(allv[a], allv[b])) for a in names for b in names if a < b],
              "S2ab_econ_overlap")
    save([graph_row(k, v) for k, v in {**ECON_VAR, **PRE}.items()], "S2b_econ_variants_graph")

In [ ]:
# Sel 6 — Table 2, S3a, S3b: variasi tersisa dan diagnostik identifikasi
VARS = {"poverty": Y, **{v: X[:, :, j] for j, v in enumerate(COL_X)}}
if HDI is not None:
    VARS["hdi"] = HDI
rows = []
for v, A in VARS.items():
    tot = A - A.mean(); w1 = A - A.mean(0, keepdims=True); w2 = w1 - w1.mean(1, keepdims=True)
    rows.append(dict(variable=v, mean=A.mean(), sd_total=A.std(ddof=1), sd_between=A.mean(0).std(ddof=1),
                     sd_within_NTm1=np.sqrt((w1 ** 2).sum() / (N * T_ALL - 1)),
                     sd_within_dof=np.sqrt((w1 ** 2).sum() / (N * (T_ALL - 1))),
                     sd_within_twoway_dof=np.sqrt((w2 ** 2).sum() / ((N - 1) * (T_ALL - 1))),
                     share_left_unit_pct=100 * (w1 ** 2).sum() / (tot ** 2).sum(),
                     share_left_twoway_pct=100 * (w2 ** 2).sum() / (tot ** 2).sum(),
                     within_common_pct=100 * (1 - (w2 ** 2).sum() / (w1 ** 2).sum())))
T2 = save(rows, "T2_S3a_variance")
print(T2.round(3).to_string(index=False))

def r2_year(A):
    w1 = A - A.mean(0, keepdims=True); w2 = w1 - w1.mean(1, keepdims=True)
    return 1 - (w2 ** 2).sum() / (w1 ** 2).sum()

def partial_F(Yt, W, fe):
    WX = lagW(W, X); Zc = np.concatenate([X, WX], axis=2)
    inst = np.concatenate([lagW(W, WX), lagW(W, lagW(W, WX))], axis=2)
    wy = demean(lagW(W, Yt), fe)[:, 0]; zc = demean(Zc, fe); zi = demean(inst, fe)
    def ssr(M):
        b = np.linalg.lstsq(M, wy, rcond=None)[0]; e = wy - M @ b; return e @ e
    s_r, s_u = ssr(zc), ssr(np.c_[zc, zi])
    q = zi.shape[1]; df2 = neff_of(T_ALL, N, fe) - zc.shape[1] - q
    return ((s_r - s_u) / q) / (s_u / df2)

S3b = save([dict(W=WLAB[k], R2_year_on_unit_demeaned_Wy_pct=100 * r2_year(lagW(ALLW[k], Y)),
                 R2_year_on_unit_demeaned_y_pct=100 * r2_year(Y),
                 partialF_W2X_W3X_unit=partial_F(Y, ALLW[k], "ind"),
                 partialF_W2X_W3X_twoway=partial_F(Y, ALLW[k], "tw")) for k in CORE], "S3b_identification")
print(S3b.round(2).to_string(index=False))

In [ ]:
# Sel 7 — Model statis: Table 4, 5 dan S4, S6–S10, S14–S23
FITS, DRAWS = {}, {}
fit_rows, coef_rows, imp_rows = [], [], []
isub = [YEARS.index(y) for y in SUB_YEARS]
XH = np.concatenate([X, HDI[..., None]], axis=2) if HDI is not None else None

def run(key, Yt, Xc, W, ev, fe, model, label):
    f = estimate(Yt, Xc, W, ev, fe, model)
    FITS[key] = f
    fit_rows.append(dict(spec=label, W=key[1], FE=fe, model=model, sample=key[3], logLik=f["ll"], k=f["k"], AIC=f["aic"]))
    coef_rows.extend(coef_table(f, label))
    r, dr = impact_table(f, ev, rng, label)
    for x in r:
        x.update(W=key[1], FE=fe, model=model, sample=key[3])
    imp_rows.extend(r); DRAWS[key] = dr
    return f

t0 = time.time()
for wk in EXT:
    for fe in ["ind", "tw"]:
        for m in ["SDM", "SAR", "SLX", "SEM", "SDEM"]:
            run((m, wk, fe, "full"), Y, X, ALLW[wk], EV[wk], fe, m, f"{m} | {WLAB[wk]} | {fe} | 2020-2024")
        if wk in CORE:
            run(("SDM", wk, fe, "2022-2024"), Y[isub], X[isub], ALLW[wk], EV[wk], fe, "SDM",
                f"SDM | {WLAB[wk]} | {fe} | 2022-2024")
            if XH is not None:
                run(("SDM", wk, fe, "full+HDI"), Y, XH, ALLW[wk], EV[wk], fe, "SDM", f"SDM+HDI | {WLAB[wk]} | {fe}")
for wk in list(ECON_VAR) + list(PRE):
    Wv = {**ECON_VAR, **PRE}[wk]
    for fe in ["ind", "tw"]:
        for m in ["SDM", "SAR"]:
            run((m, wk, fe, "full"), Y, X, Wv, EV[wk], fe, m, f"{m} | {wk} | {fe}")
print(f"{len(FITS)} model diestimasi dalam {time.time()-t0:.0f}s")
S6 = save(fit_rows, "S6_model_fit")
S7 = save(coef_rows, "S7_coefficients")
S8 = save(imp_rows, "S8_impacts")

# Table 4: queen SDM, kedua FE
T4 = S8[(S8.W == "queen") & (S8.model == "SDM") & (S8["sample"] == "full")]
save(T4, "T4_queen_SDM_impacts")
save(S7[S7.spec.str.startswith("SDM | Queen contiguity") & S7.spec.str.endswith("2020-2024")], "T4_queen_SDM_coefficients")
print(T4[T4.variable.isin(["manuf", "sanit"])].round(3).to_string(index=False))

# S9: indirect share of total (sanitasi), SDM
js = COL_X.index("sanit")
s9 = []
for (m, wk, fe, smp), dr in DRAWS.items():
    if m == "SDM" and smp == "full" and wk in EXT:
        ratio = dr[:, js, 1] / dr[:, js, 2]
        pt = impacts_draws(FITS[(m, wk, fe, smp)], EV[wk], 1, rng, point=True)[0]
        s9.append(dict(W=WLAB[wk], FE=fe, share_point=pt[js, 1] / pt[js, 2],
                       lwr95=np.quantile(ratio, .025), upr95=np.quantile(ratio, .975)))
save(s9, "S9_indirect_share_sanit")

# S10: rho = 0 substitusi vs SLX diestimasi ulang
s10 = []
for wk in CORE:
    for fe in ["ind", "tw"]:
        sdm, slx = FITS[("SDM", wk, fe, "full")], FITS[("SLX", wk, fe, "full")]
        K = sdm["K"]
        for j, v in enumerate(COL_X):
            s10.append(dict(W=WLAB[wk], FE=fe, variable=v, indirect_substituted=sdm["theta"][1 + K + j],
                            indirect_SLX=slx["theta"][1 + K + j],
                            se_SLX=np.sqrt(slx["cov"][1 + K + j, 1 + K + j])))
save(s10, "S10_rho0_substitution_vs_SLX")

# S16: LR tests conditional on W
lr = []
for wk in EXT:
    for fe in ["ind", "tw"]:
        g = lambda m: FITS[(m, wk, fe, "full")]["ll"]
        for alt, df_ in [("SAR", K), ("SLX", 1), ("SEM", K)]:
            st = 2 * (g("SDM") - g(alt))
            lr.append(dict(W=WLAB[wk], FE=fe, test=f"SDM vs {alt}", LR=st, df=df_, p=chi2.sf(st, df_)))
        st = 2 * (g("SDEM") - g("SEM"))
        lr.append(dict(W=WLAB[wk], FE=fe, test="SDEM vs SEM", LR=st, df=K, p=chi2.sf(st, K)))
S16 = save(lr, "S16_LR_tests")

# S17: LM dan robust LM (Anselin et al. 1996; Elhorst 2014), model non-spasial FE
lm = []
for wk in CORE:
    W = ALLW[wk]
    for fe in ["ind", "tw"]:
        Tn = T_ALL; xt = demean(X, fe); yt = demean(Y, fe)[:, 0]
        b = np.linalg.lstsq(xt, yt, rcond=None)[0]; e = yt - xt @ b
        s2 = e @ e / neff_of(Tn, N, fe)
        Wy = demean(lagW(W, Y), fe)[:, 0]; We = lagW(W, e.reshape(Tn, N)).reshape(-1)
        WXb = lagW(W, (xt @ b).reshape(Tn, N)).reshape(-1)
        Mx = WXb - xt @ np.linalg.lstsq(xt, WXb, rcond=None)[0]
        TW = np.trace(W.T @ W + W @ W)
        J = (Mx @ Mx + Tn * TW * s2) / s2
        gl, ge = e @ Wy / s2, e @ We / s2
        vals = dict(LM_lag=gl ** 2 / J, LM_error=ge ** 2 / (Tn * TW),
                    robust_LM_lag=(gl - ge) ** 2 / (J - Tn * TW),
                    robust_LM_error=(ge - Tn * TW / J * gl) ** 2 / (Tn * TW * (1 - Tn * TW / J)))
        for kk, vv in vals.items():
            lm.append(dict(W=WLAB[wk], FE=fe, test=kk, stat=vv, p=chi2.sf(vv, 1)))
save(lm, "S17_LM_tests")

# S18a: Hausman non-spasial (FE vs Swamy-Arora RE)
xw = demean(X, "ind"); yw = demean(Y, "ind")[:, 0]
bfe = np.linalg.lstsq(xw, yw, rcond=None)[0]; ew = yw - xw @ bfe
s2e = ew @ ew / (N * (T_ALL - 1) - K)
xb = np.c_[np.ones(N), X.mean(0)]; yb = Y.mean(0)
eb = yb - xb @ np.linalg.lstsq(xb, yb, rcond=None)[0]
s2u = max(eb @ eb / (N - K - 1) - s2e / T_ALL, 0)
th_ = 1 - np.sqrt(s2e / (T_ALL * s2u + s2e))
xr = np.c_[np.full(N * T_ALL, 1 - th_), (X - th_ * X.mean(0, keepdims=True)).reshape(-1, K)]
yr = (Y - th_ * Y.mean(0, keepdims=True)).reshape(-1)
bre = np.linalg.lstsq(xr, yr, rcond=None)[0]; er = yr - xr @ bre
Vfe = s2e * np.linalg.inv(xw.T @ xw); Vre = (er @ er / (N * T_ALL - K - 1)) * np.linalg.inv(xr.T @ xr)[1:, 1:]
d = bfe - bre[1:]
H = float(d @ np.linalg.pinv(Vfe - Vre) @ d)
save([dict(test="Hausman FE vs RE (Swamy-Arora)", chi2=H, df=K, p=chi2.sf(H, K), theta=th_)], "S18a_hausman")

# S19: selisih AIC berpasangan (SDM)
s19 = []
for fe in ["ind", "tw"]:
    for a in EXT:
        for b_ in EXT:
            if a < b_:
                s19.append(dict(FE=fe, W_a=WLAB[a], W_b=WLAB[b_],
                                dAIC_b_minus_a=FITS[("SDM", b_, fe, "full")]["aic"] - FITS[("SDM", a, fe, "full")]["aic"]))
save(s19, "S19_pairwise_AIC")

# S4: interval profile-likelihood untuk rho (SDM)
s4 = []
for wk in EXT:
    for fe in ["ind", "tw"]:
        Z = FITS[("SDM", wk, fe, "full")]["Z"]; T, n = Y.shape
        zt = demean(Z, fe); yt = demean(Y, fe)[:, 0]; wyt = demean(lagW(ALLW[wk], Y), fe)[:, 0]
        P = np.linalg.solve(zt.T @ zt, zt.T); e0 = yt - zt @ (P @ yt); e1 = wyt - zt @ (P @ wyt)
        grid = np.linspace(LO, HI, 4000)
        sse = e0 @ e0 - 2 * grid * (e0 @ e1) + grid ** 2 * (e1 @ e1)
        ll = -0.5 * neff_of(T, n, fe) * (np.log(2 * np.pi * sse / neff_of(T, n, fe)) + 1) + \
             np.array([ldet(r, EV[wk], T, fe) for r in grid])
        inside = grid[2 * (ll.max() - ll) <= chi2.ppf(.95, 1)]
        s4.append(dict(W=WLAB[wk], FE=fe, rho_hat=grid[ll.argmax()], lwr95=inside.min(), upr95=inside.max()))
save(s4, "S4_profile_rho")

# S23b / S23a: VIF dan korelasi pada desain terdemean [X, WX]
def vif(M):
    out = []
    for j in range(M.shape[1]):
        o = np.delete(M, j, 1); b = np.linalg.lstsq(o, M[:, j], rcond=None)[0]; e = M[:, j] - o @ b
        out.append((M[:, j] @ M[:, j]) / (e @ e))
    return out
s23 = []
for wk in EXT + list(PRE):
    Wv = {**ALLW, **PRE}[wk]
    for fe in ["ind", "tw"]:
        M = demean(np.concatenate([X, lagW(Wv, X)], axis=2), fe)
        v = vif(M)
        for j, nm in enumerate([f"{c}" for c in COL_X] + [f"W_{c}" for c in COL_X]):
            s23.append(dict(W=WLAB.get(wk, wk), FE=fe, term=nm, VIF=v[j],
                            corr_X_WX=np.corrcoef(M[:, j % K], M[:, K + j % K])[0, 1]))
save(s23, "S23ab_VIF_demeaned_design")

# S20-S22: varian matriks ekonomi dan pra-sampel
s20 = []
for wk in ["econ"] + list(ECON_VAR) + list(PRE):
    for fe in ["ind", "tw"]:
        sdm, sar = FITS[("SDM", wk, fe, "full")], FITS[("SAR", wk, fe, "full")]
        st = 2 * (sdm["ll"] - sar["ll"])
        row = S8[(S8.W == wk) & (S8.FE == fe) & (S8.model == "SDM") & (S8.variable == "sanit") & (S8.effect == "indirect")].iloc[0]
        s20.append(dict(W=WLAB.get(wk, wk), FE=fe, rho=sdm["theta"][0], sanit_indirect=row.estimate,
                        lwr95=row.lwr95, upr95=row.upr95, significant=(row.lwr95 > 0) | (row.upr95 < 0),
                        joint_Durbin_LR=st, joint_Durbin_p=chi2.sf(st, K), AIC=sdm["aic"]))
save(s20, "S20_S22_econ_variants")

# Table 5: indirect effect sanitasi lintas model, W, FE, HDI, subsampel
T5 = S8[(S8.variable == "sanit") & (S8.effect == "indirect") & S8.model.isin(["SDM", "SLX", "SDEM"])
        & S8.W.isin(EXT)]
T5 = save(T5[["model", "W", "FE", "sample", "estimate", "lwr95", "upr95"]], "T5_sanit_indirect_robustness")
print(T5.round(3).to_string(index=False))

In [ ]:
# Sel 8 — Table 3, 6 dan S11–S13: elpd (PSIS-LOO dengan exact refit, WAIC), bobot model, model averaging
RGRID = np.linspace(LO, HI, 800)
def safe_chol(M):
    w, V = np.linalg.eigh((M + M.T) / 2)
    return V * np.sqrt(np.clip(w, 0, None))

class BayesSDM:
    """SDM statis Bayes: prior datar untuk koefisien dan efek tetap, p(sigma2) ~ 1/sigma2.
    Sampling terkolaps: rho dari posterior marginal (grid), lalu sigma2, beta, efek tetap."""
    def __init__(self, Yt, Z, W, ev, fe):
        self.T, self.n = Yt.shape; self.W, self.fe, self.Z = W, fe, Z
        self.zt = demean(Z, fe); self.P = np.linalg.solve(self.zt.T @ self.zt, self.zt.T)
        self.ZtZi = np.linalg.inv(self.zt.T @ self.zt); self.Lz = safe_chol(self.ZtZi)
        self.neff = neff_of(self.T, self.n, fe); self.k = Z.shape[2]
        self.ld = np.array([ldet(r, ev, self.T, fe) for r in RGRID])
        D = np.kron(np.ones((self.T, 1)), np.eye(self.n))
        if fe == "tw":
            D = np.hstack([D, np.kron(np.eye(self.T)[:, 1:], np.ones((self.n, 1)))])
        self.D = D; self.DtDi = np.linalg.inv(D.T @ D); self.Ld = safe_chol(self.DtDi)
        self.Qii = 1 + RGRID[:, None] ** 2 * (W ** 2).sum(0)[None, :]      # (A'A)_ii per rho
        self.set_y(Yt)
    def set_y(self, Yt):
        self.Y = Yt; self.WY = lagW(self.W, Yt)
        yt = demean(Yt, self.fe)[:, 0]; wyt = demean(self.WY, self.fe)[:, 0]
        self.b0, self.b1 = self.P @ yt, self.P @ wyt
        e0, e1 = yt - self.zt @ self.b0, wyt - self.zt @ self.b1
        self.q = (e0 @ e0, e0 @ e1, e1 @ e1)
    def draw(self, rr):
        a, c, e = self.q
        sse = a - 2 * RGRID * c + RGRID ** 2 * e
        lp = self.ld - 0.5 * (self.neff - self.k) * np.log(sse)
        p = np.exp(lp - lp.max()); p /= p.sum()
        j = rr.choice(len(RGRID), p=p)
        r = RGRID[j]
        s2 = sse[j] / rr.chisquare(self.neff - self.k)
        b = self.b0 - r * self.b1 + np.sqrt(s2) * self.Lz @ rr.standard_normal(self.k)
        res = (self.Y - r * self.WY - self.Z @ b).reshape(-1)
        c_hat = self.DtDi @ (self.D.T @ res)
        fe_ = c_hat + np.sqrt(s2) * self.Ld @ rr.standard_normal(len(c_hat))
        e_ = (res - self.D @ fe_).reshape(self.T, self.n)
        return j, r, s2, b, e_
    def pointwise(self, j, r, s2, e_):
        Ae = e_ - r * lagW(self.W.T, e_)                       # A'e per tahun
        Q = self.Qii[j][None, :]
        return -0.5 * np.log(2 * np.pi * s2 / Q) - 0.5 * Ae ** 2 / (s2 * Q)

def gpdfit(x):
    n = len(x); prior_bs, prior_k = 3, 10
    m = 30 + int(np.sqrt(n))
    bs = 1 - np.sqrt(m / (np.arange(1, m + 1) - 0.5))
    bs /= prior_bs * x[int(n / 4 + 0.5) - 1]
    bs += 1 / x[-1]
    ks = np.log1p(-bs[:, None] * x).mean(axis=1)
    L = n * (np.log(-(bs / ks)) - ks - 1)
    w = 1 / np.exp(L - L[:, None]).sum(axis=1)
    ok = w >= 10 * np.finfo(float).eps; w = w[ok] / w[ok].sum(); bs = bs[ok]
    b = (bs * w).sum(); k = np.log1p(-b * x).mean(); sigma = -k / b
    return (n * k + prior_k * 0.5) / (n + prior_k), sigma

def psis_loo(ll):
    """ll: (S, n) pointwise log-lik. Returns elpd_i, k_hat_i (PSIS, Vehtari et al. 2024)."""
    S, n = ll.shape; M = int(np.ceil(min(0.2 * S, 3 * np.sqrt(S))))
    elpd, khat = np.empty(n), np.empty(n)
    for i in range(n):
        lw = -ll[:, i]; lw = lw - lw.max()
        o = np.argsort(lw); cut = lw[o[-M - 1]]
        x = np.exp(lw[o[-M:]]) - np.exp(cut)
        k, sig = gpdfit(np.sort(x))
        p = (np.arange(M) + 0.5) / M
        tail = sig * np.expm1(-k * np.log1p(-p)) / k if abs(k) > 1e-12 else -sig * np.log1p(-p)
        lw[o[-M:]] = np.minimum(np.log(tail + np.exp(cut)), 0)
        lw -= logsumexp(lw)
        elpd[i] = logsumexp(lw + ll[:, i]); khat[i] = k
    return elpd, khat

def waic(ll):
    lppd = logsumexp(ll, axis=0) - np.log(len(ll)); p = ll.var(0, ddof=1)
    return lppd - p

def exact_loo(Yt, Z, W, ev, fe, t, i, seed):
    """Exact leave-one-out: refit dengan y_it diperlakukan sebagai missing (data augmentation)."""
    rr = np.random.default_rng(seed)
    m = BayesSDM(Yt, Z, W, ev, fe); yobs = Yt[t, i]; Yc = Yt.copy(); dens = []
    for it in range(REFIT_ITER):
        j, r, s2, b, e_ = m.draw(rr)
        Ae = e_ - r * lagW(W.T, e_)
        Q = m.Qii[j][i]
        cm = Yc[t, i] - Ae[t, i] / Q                              # mean bersyarat y_it | y_-i,t
        if it >= REFIT_BURN:
            dens.append(norm.logpdf(yobs, cm, np.sqrt(s2 / Q)))
        Yc[t, i] = cm + np.sqrt(s2 / Q) * rr.standard_normal()
        m.set_y(Yc)
    return logsumexp(dens) - np.log(len(dens))

LL, ELPD = {}, {}
s11, s11c = [], []
for wk in EXT:
    for fe in ["ind", "tw"]:
        Z = FITS[("SDM", wk, fe, "full")]["Z"]
        m = BayesSDM(Y, Z, ALLW[wk], EV[wk], fe); rr = np.random.default_rng([SEED, 11])
        ll = np.empty((BAYES_S, N * T_ALL))
        for s in range(BAYES_S):
            j, r, s2, b, e_ = m.draw(rr); ll[s] = m.pointwise(j, r, s2, e_).reshape(-1)
        loo_i, kh = psis_loo(ll); w_i = waic(ll); loo_psis = loo_i.copy()
        bad = np.where(kh > 0.7)[0]
        res = Parallel(n_jobs=N_JOBS)(delayed(exact_loo)(Y, Z, ALLW[wk], EV[wk], fe, b_ // N, b_ % N, [SEED, 12, b_])
                                      for b_ in bad)
        for b_, v in zip(bad, res):
            s11c.append(dict(W=WLAB[wk], FE=fe, unit=ids[b_ % N], year=YEARS[b_ // N], pareto_k=kh[b_],
                             elpd_i_psis=loo_i[b_], elpd_i_exact=v, difference=v - loo_i[b_]))
            loo_i[b_] = v
        LL[(wk, fe)] = ll; ELPD[(wk, fe)] = dict(loo=loo_i, waic=w_i, psis=loo_psis, k=kh, n_refit=len(bad))
        print(f"{wk:9s} {fe:3s} elpd_loo {loo_i.sum():8.2f} (PSIS {loo_psis.sum():8.2f}, refit {len(bad)}) "
              f"waic {w_i.sum():8.2f} max k {kh.max():.2f}")
save(s11c, "S11c_exact_refits_pointwise")

def compare(set_name, keys):
    rows = []
    for fe in ["ind", "tw"]:
        for crit in ["loo", "waic"]:
            tot = {k: ELPD[(k, fe)][crit].sum() for k in keys}
            best = max(tot, key=tot.get)
            for k in keys:
                d = ELPD[(k, fe)][crit] - ELPD[(best, fe)][crit]
                se_d = np.sqrt(len(d) * d.var(ddof=1)) if k != best else 0.0
                rows.append(dict(set=set_name, FE=fe, criterion=crit, W=WLAB[k], elpd=tot[k],
                                 se=np.sqrt(len(d) * ELPD[(k, fe)][crit].var(ddof=1)), diff=d.sum(), se_diff=se_d,
                                 share_k_gt_0_7=(ELPD[(k, fe)]["k"] > 0.7).mean(), max_k=ELPD[(k, fe)]["k"].max(),
                                 n_refit=ELPD[(k, fe)]["n_refit"], elpd_psis_only=ELPD[(k, fe)]["psis"].sum()))
            rv = [r for r in rows if r["FE"] == fe and r["criterion"] == crit and r["diff"] != 0]
            z = min(abs(r["diff"]) / r["se_diff"] for r in rv)
            rows.append(dict(set=set_name, FE=fe, criterion=crit, W="DECISION", elpd=np.nan,
                             diff=z, se_diff=np.nan, best=WLAB[best],
                             decision="SELECT " + WLAB[best] if z > 2 else "not separated -> average over W"))
    return rows
S11 = save(compare("core", CORE) + (compare("extended", EXT) if len(EXT) > 3 else []), "T3_S11_elpd_comparison")
print(S11[S11.W == "DECISION"][["set", "FE", "criterion", "diff", "decision"]].to_string(index=False))

# Bobot: Akaike, stacking (Yao et al. 2018), BMA g-prior (Fernández et al. 2001)
def stacking(keys, fe):
    L = np.column_stack([ELPD[(k, fe)]["loo"] for k in keys])
    f = lambda z: -logsumexp(L + (z - logsumexp(z))[None, :], axis=1).sum()
    z = minimize(f, np.zeros(len(keys)), method="BFGS").x
    return np.exp(z - logsumexp(z))

def log_marg(wk, fe, model):
    Z = FITS[("SDM", wk, fe, "full")]["Z"]; T, n = Y.shape; ne = neff_of(T, n, fe)
    zt = demean(Z, fe); yt = demean(Y, fe)[:, 0]; wyt = demean(lagW(ALLW[wk], Y), fe)[:, 0]
    g = ne; k = zt.shape[1]; H = zt @ np.linalg.solve(zt.T @ zt, zt.T)
    def lm(r):
        yr = yt - r * wyt
        return -0.5 * k * np.log(1 + g) - 0.5 * ne * np.log(yr @ yr - g / (1 + g) * yr @ H @ yr)
    if model == "SLX":
        return lm(0.0)
    v = np.array([lm(r) + ldet(r, EV[wk], T, fe) for r in RGRID])
    return logsumexp(v) - np.log(len(v))

s13a = []; WEIGHTS = {}
for set_name, keys in [("core", CORE)] + ([("extended", EXT)] if len(EXT) > 3 else []):
    for fe in ["ind", "tw"]:
        aic = np.array([FITS[("SDM", k, fe, "full")]["aic"] for k in keys])
        ak = np.exp(-0.5 * (aic - aic.min())); ak /= ak.sum()
        st = stacking(keys, fe)
        lsdm = np.array([log_marg(k, fe, "SDM") for k in keys]); lslx = np.array([log_marg(k, fe, "SLX") for k in keys])
        pw = np.exp(lsdm - logsumexp(lsdm))
        allm = np.r_[lsdm, lslx]; pm = np.exp(allm - logsumexp(allm))
        for i, k in enumerate(keys):
            s13a.append(dict(set=set_name, FE=fe, W=WLAB[k], akaike=ak[i], stacking_loo=st[i], bma_P_W_given_SDM=pw[i],
                             bma_P_W_marginal_SDM_SLX=pm[i] + pm[len(keys) + i],
                             bma_P_SDM_given_W=pm[i] / (pm[i] + pm[len(keys) + i])))
        WEIGHTS[(set_name, fe)] = dict(Akaike=ak, Stacking=st, BMA=pw, keys=keys)
S13a = save(s13a, "S12_S13a_model_weights")
print(S13a.round(3).to_string(index=False))

# Table 6 / S13b: model averaging (campuran draws per W sebanding bobot)
s13b = []
for (set_name, fe), wd in WEIGHTS.items():
    for scheme in ["Akaike", "Stacking", "BMA"]:
        w = wd[scheme]; keys = wd["keys"]
        pts = np.array([impacts_draws(FITS[("SDM", k, fe, "full")], EV[k], 1, rng, point=True)[0] for k in keys])
        pt = (w[:, None, None] * pts).sum(0)
        nk = np.round(w * N_MIX).astype(int)
        mix = np.concatenate([DRAWS[("SDM", k, fe, "full")][rng.integers(0, N_DRAWS, n_)] for k, n_ in zip(keys, nk) if n_ > 0])
        lo, hi = np.quantile(mix, [.025, .975], axis=0)
        for j, v in enumerate(COL_X):
            for e, en in enumerate(["direct", "indirect", "total"]):
                s13b.append(dict(set=set_name, FE=fe, weights=scheme, variable=v, effect=en,
                                 estimate=pt[j, e], lwr95=lo[j, e], upr95=hi[j, e],
                                 weights_used=" / ".join(f"{x:.2f}" for x in w)))
S13b = save(s13b, "T6_S13b_model_averaged_impacts")
print(S13b[(S13b.variable == "sanit") & (S13b.effect.isin(["indirect", "total"]))].round(3).to_string(index=False))

In [ ]:
# Sel 9 — Monte Carlo: Table S5 (rho dengan/tanpa guncangan tahun) dan S32 (pemulihan W yang benar)
def sim_static(W, theta, K_, FEm, s2, rr):
    r, b = theta[0], theta[1:1 + 2 * K_]
    Z = np.concatenate([X, lagW(W, X)], axis=2)
    A = np.linalg.inv(np.eye(N) - r * W)
    return np.einsum("ij,tj->ti", A, Z @ b + FEm + np.sqrt(s2) * rr.standard_normal((T_ALL, N)))

def fe_static(W, theta, K_, fe):
    Z = np.concatenate([X, lagW(W, X)], axis=2)
    R = Y - theta[0] * lagW(W, Y) - Z @ theta[1:1 + 2 * K_]
    mu = R.mean(0, keepdims=True)
    return (mu + R.mean(1, keepdims=True) - R.mean()) if fe == "tw" else np.repeat(mu, T_ALL, 0)

if RUN_MC:
    # S5: DGP = SDM queen two-way FE (dengan efek tahun) atau tanpa efek tahun
    Wq, evq = ALLW["queen"], EV["queen"]
    f_tw = FITS[("SDM", "queen", "tw", "full")]
    FE_tw = fe_static(Wq, f_tw["theta"], K, "tw")
    FE_unit_only = np.repeat(FE_tw.mean(0, keepdims=True), T_ALL, 0)
    def s5_one(r_, with_year):
        rr = np.random.default_rng([SEED, 5, r_, int(with_year)])
        Ys = sim_static(Wq, f_tw["theta"], K, FE_tw if with_year else FE_unit_only, f_tw["s2"], rr)
        out = []
        for fe in ["ind", "tw"]:
            f = estimate(Ys, X, Wq, evq, fe, "SDM"); se = np.sqrt(f["cov"][0, 0])
            out.append(dict(year_effects_in_DGP=with_year, FE=fe, rho=f["theta"][0],
                            cover=abs(f["theta"][0] - f_tw["theta"][0]) <= 1.96 * se))
        return out
    t0 = time.time()
    res = Parallel(n_jobs=N_JOBS)(delayed(s5_one)(r_, wy) for wy in [True, False] for r_ in range(MC_RHO_REPS))
    s5 = pd.DataFrame([x for r_ in res for x in r_])
    S5 = s5.groupby(["year_effects_in_DGP", "FE"]).agg(mean_rho=("rho", "mean"), coverage95=("cover", "mean")).reset_index()
    S5["true_rho"] = f_tw["theta"][0]; S5["bias"] = S5.mean_rho - S5.true_rho
    save(S5, "S5_MC_rho_year_shocks"); print(S5.round(3).to_string(index=False), f"({time.time()-t0:.0f}s)")

    # S32: data dari tiap SDM unit-FE; AIC dan probabilitas posterior memilih W
    def s32_one(r_, true_w):
        rr = np.random.default_rng([SEED, 32, r_, CORE.index(true_w)])
        f0 = FITS[("SDM", true_w, "ind", "full")]
        Ys = sim_static(ALLW[true_w], f0["theta"], K, fe_static(ALLW[true_w], f0["theta"], K, "ind"), f0["s2"], rr)
        aic, lm_ = [], []
        for wk in CORE:
            Z = np.concatenate([X, lagW(ALLW[wk], X)], axis=2)
            r = fit_lag(Ys, Z, ALLW[wk], EV[wk], "ind"); aic.append(-2 * r["ll"])
            zt = demean(Z, "ind"); yt = demean(Ys, "ind")[:, 0]; wyt = demean(lagW(ALLW[wk], Ys), "ind")[:, 0]
            ne = neff_of(T_ALL, N, "ind"); g = ne; H = zt @ np.linalg.solve(zt.T @ zt, zt.T)
            v = [-0.5 * ne * np.log((yt - q * wyt) @ (yt - q * wyt) - g / (1 + g) * (yt - q * wyt) @ H @ (yt - q * wyt))
                 + ldet(q, EV[wk], T_ALL, "ind") for q in RGRID[::8]]
            lm_.append(logsumexp(v))
        return dict(true_W=true_w, AIC_choice=CORE[int(np.argmin(aic))], BMA_choice=CORE[int(np.argmax(lm_))])
    t0 = time.time()
    res = Parallel(n_jobs=N_JOBS)(delayed(s32_one)(r_, tw_) for tw_ in CORE for r_ in range(MC_W_REPS))
    s32 = pd.DataFrame(res)
    S32 = s32.groupby("true_W").agg(AIC_correct=("AIC_choice", lambda s: (s == s.name).mean()),
                                    reps=("AIC_choice", "size")).reset_index()
    S32["AIC_correct"] = [(s32[s32.true_W == w].AIC_choice == w).mean() for w in S32.true_W]
    S32["BMA_correct"] = [(s32[s32.true_W == w].BMA_choice == w).mean() for w in S32.true_W]
    save(S32, "S32_MC_W_recovery"); print(S32.round(3).to_string(index=False), f"({time.time()-t0:.0f}s)")
else:
    print("RUN_MC = False: Table S5 dan S32 dilewati.")

In [ ]:
# Sel 10 — SDM dinamis: Table 7, 8 dan S26–S31
def fe_given(Yt, Z, W, d, b, fe):
    R = Yt - d * lagW(W, Yt) - Z @ b
    mu = R.mean(0, keepdims=True)
    if fe == "tw":
        return mu + R.mean(1, keepdims=True) - R.mean()
    return np.repeat(mu, R.shape[0], 0)

def resid_s2(Yt, Z, W, d, b, fe):
    R = Yt - d * lagW(W, Yt) - Z @ b - fe_given(Yt, Z, W, d, b, fe)
    T, n = Yt.shape
    return (R ** 2).sum() / neff_of(T, n, fe)

def dyn_design(Yfull, W):
    yl = Yfull[:-1][..., None]
    WX = lagW(W, X)
    Z = np.concatenate([yl, lagW(W, yl), X[1:], WX[1:], X[:-1], WX[:-1]], axis=2)
    return Yfull[1:], Z

NAMES = (["tau", "eta"] + [f"b_{c}" for c in COL_X] + [f"th_{c}" for c in COL_X]
         + [f"bl_{c}" for c in COL_X] + [f"thl_{c}" for c in COL_X])

def fit_dyn(Yfull, W, ev, fe):
    Yt, Z = dyn_design(Yfull, W)
    r = fit_lag(Yt, Z, W, ev, fe)
    return np.r_[r["d"], r["b"]]          # [delta, tau, eta, ...]

def spec_rad(p, ev):
    d, tau, eta = p[0], p[1], p[2]
    return np.max(np.abs((tau + eta * ev) / (1 - d * ev)))

def sim_dyn(p, FE, s, W, eps):
    d, tau, eta, beta = p[0], p[1], p[2], p[3:]
    A = np.linalg.inv(np.eye(N) - d * W)
    WX = lagW(W, X)
    Ys = np.empty_like(Y); Ys[0] = Y[0]
    for t in range(1, T_ALL):
        xb = np.concatenate([X[t], WX[t], X[t-1], WX[t-1]], axis=1) @ beta
        Ys[t] = A @ (tau * Ys[t-1] + eta * (W @ Ys[t-1]) + xb + FE[t-1] + s * eps[t-1])
    return Ys

def implied(p, W, fe, Yfull=None):
    Yfull = Y if Yfull is None else Yfull
    Yt, Z = dyn_design(Yfull, W)
    FE = fe_given(Yt, Z, W, p[0], p[1:], fe)
    s = np.sqrt(resid_s2(Yt, Z, W, p[0], p[1:], fe))
    return FE, s
def boot_fits(p, FE, s, W, ev, fe, E):
    return np.array([fit_dyn(sim_dyn(p, FE, s, W, e), W, ev, fe) for e in E])

def bc_single(p_hat, W, ev, fe, E, Yfull=None):
    FE, s = implied(p_hat, W, fe, Yfull)
    draws = boot_fits(p_hat, FE, s, W, ev, fe, E)
    return 2 * p_hat - draws.mean(0), draws

def bc_iter(p_hat, W, ev, fe, E, Yfull=None):
    p = p_hat.copy(); hist = []
    for it in range(MAX_ITER):
        FE, s = implied(p, W, fe, Yfull)
        draws = boot_fits(p, FE, s, W, ev, fe, E)
        gap = p_hat - draws.mean(0)
        hist.append(np.max(np.abs(gap[:3])))
        if hist[-1] < TOL:
            break
        step = 1.0
        while True:
            cand = p + step * gap
            if -0.99 < cand[0] < 0.999 and spec_rad(cand, ev) < 0.995:
                break
            step *= 0.5
            if step < 1e-3:
                cand = p
                break
        p = cand
    return p, draws, it + 1, hist[-1]

def basic_interval(p_hat, p_gen, draws, ev, level=0.95):
    """Basic bootstrap. p_gen = parameter at which the bootstrap samples were generated:
    single-step -> p_hat (interval centred on 2*p_hat - mean); iterated -> converged value."""
    pseudo = p_hat + p_gen - draws
    a = (1 - level) / 2
    lo, hi = np.quantile(pseudo, [a, 1 - a], axis=0)
    sr = np.array([spec_rad(q, ev) for q in pseudo])
    return lo, hi, pseudo.std(0, ddof=1), sr
DGRID = np.linspace(-0.99, 0.999, 400)
def fit_bayes_dyn(Yfull, W, ev, mode, n_iter, burn, seed):
    """mode: fe_unit, fe_tw, re (efek regency acak), pooled. Prior N(0,100^2), IG(0.01,0.01), delta ~ U."""
    rr = np.random.default_rng(seed)
    Yt, Z = dyn_design(Yfull, W); T, n = Yt.shape
    y = Yt.reshape(-1); wy = lagW(W, Yt).reshape(-1); Zl = Z.reshape(T * n, -1); kz = Zl.shape[1]
    Du = np.kron(np.ones((T, 1)), np.eye(n))
    if mode == "fe_unit":
        Xf = np.hstack([Zl, Du])
    elif mode == "fe_tw":
        Xf = np.hstack([Zl, Du, np.kron(np.eye(T)[:, 1:], np.ones((n, 1)))])
    elif mode == "re":
        Xf = np.hstack([Zl, np.ones((T * n, 1)), Du])
    else:
        Xf = np.hstack([Zl, np.ones((T * n, 1))])
    k = Xf.shape[1]; XtX, Xty, Xtw = Xf.T @ Xf, Xf.T @ y, Xf.T @ wy
    ld = T * np.array([np.sum(np.log(np.abs(1 - d * ev))).real for d in DGRID])
    pri = np.full(k, 1e-4); d = 0.0; s2 = 1.0; s2mu = 1.0; keep = []
    for it in range(n_iter):
        if mode == "re":
            pri[kz + 1:] = 1 / s2mu
        prec = XtX / s2 + np.diag(pri)
        L = np.linalg.cholesky(prec)
        b = np.linalg.solve(prec, (Xty - d * Xtw) / s2) + np.linalg.solve(L.T, rr.standard_normal(k))
        r0 = y - Xf @ b; e = r0 - d * wy
        s2 = 1 / rr.gamma(0.01 + T * n / 2, 1 / (0.01 + e @ e / 2))
        if mode == "re":
            mu = b[kz + 1:]; s2mu = 1 / rr.gamma(0.01 + n / 2, 1 / (0.01 + mu @ mu / 2))
        lp = ld - (r0 @ r0 - 2 * DGRID * (r0 @ wy) + DGRID ** 2 * (wy @ wy)) / (2 * s2)
        p = np.exp(lp - lp.max()); p /= p.sum()
        d = DGRID[rr.choice(len(DGRID), p=p)] + (rr.random() - 0.5) * (DGRID[1] - DGRID[0])
        if it >= burn:
            keep.append(np.r_[d, b[:kz]])
    return np.array(keep)

# --- Table 7, S27, S28: koreksi bias satu langkah (499 bootstrap) ---
E_emp = np.random.default_rng([SEED, 600]).standard_normal((B_EMP, T_ALL - 1, N))
t7, s28, s29 = [], [], []
PN = ["delta"] + NAMES
def dyn_impacts(p, ev, W):
    """SR dan LR impacts tiap X; p = [delta, tau, eta, b(4), th(4), bl(4), thl(4)]."""
    d, tau, eta = p[0], p[1], p[2]; b, th, bl, thl = p[3:7], p[7:11], p[11:15], p[15:19]
    I_ = np.eye(N); A = np.linalg.inv(I_ - d * W); B = np.linalg.inv((1 - tau) * I_ - (d + eta) * W)
    out = []
    for j in range(K):
        S = A @ (b[j] * I_ + th[j] * W); L = B @ ((b[j] + bl[j]) * I_ + (th[j] + thl[j]) * W)
        out.append([np.trace(S) / N, S.sum() / N - np.trace(S) / N, S.sum() / N,
                    np.trace(L) / N, L.sum() / N - np.trace(L) / N, L.sum() / N])
    return np.array(out)
DYN = {}
for wk in CORE:
    W, ev = ALLW[wk], EV[wk]
    for fe in ["ind", "tw"]:
        p_hat = fit_dyn(Y, W, ev, fe)
        p_bc, draws = bc_single(p_hat, W, ev, fe, E_emp)
        pseudo = 2 * p_hat - draws                       # basic bootstrap, dibangkitkan di p_hat
        sr = np.array([spec_rad(q, ev) for q in pseudo])
        se = pseudo.std(0, ddof=1)
        DYN[(wk, fe)] = (p_hat, p_bc, pseudo)
        t7.append(dict(W=WLAB[wk], FE=fe, tau=p_bc[1], se_tau=se[1], delta=p_bc[0], se_delta=se[0],
                       eta=p_bc[2], se_eta=se[2], tau_QML=p_hat[1], spectral_radius=spec_rad(p_bc, ev),
                       sr_lwr95=np.quantile(sr, .025), sr_upr95=np.quantile(sr, .975), P_stable=(sr < 1).mean()))
        for nm, v, s_ in zip(PN, p_bc, se):
            s28.append(dict(W=WLAB[wk], FE=fe, param=nm, estimate=v, boot_se=s_, QML=p_hat[PN.index(nm)]))
        if wk == "queen" or True:
            pt = dyn_impacts(p_bc, ev, W)
            stable = pseudo[sr < 1]
            dr = np.array([dyn_impacts(q, ev, W) for q in stable[:min(len(stable), 499)]])
            lo, hi = np.quantile(dr, [.025, .975], axis=0)
            for j, v in enumerate(COL_X):
                for e, en in enumerate(["SR_direct", "SR_indirect", "SR_total", "LR_direct", "LR_indirect", "LR_total"]):
                    s29.append(dict(W=WLAB[wk], FE=fe, variable=v, effect=en, estimate=pt[j, e], lwr95=lo[j, e], upr95=hi[j, e]))
        print(f"{wk:5s} {fe:3s} tau QML {p_hat[1]:.3f} -> BC {p_bc[1]:.3f}; SR {spec_rad(p_bc, ev):.3f}")
T7 = save(t7, "T7_S27_dynamic_bc"); save(s28, "S28_dynamic_coefficients")
S29 = save(s29, "S29_dynamic_impacts")
save(S29[(S29.W == WLAB["queen"]) & (S29.variable == "sanit")], "T8_dynamic_sanit_queen")
save([dict(N=N, periods=T_ALL - 1, observations=N * (T_ALL - 1), regression_coefficients=len(NAMES),
           extra="delta, sigma2", regency_effects=N, year_effects_twoway=T_ALL - 2)], "S26_dynamic_sample")
print(T7.round(3).to_string(index=False))

# --- S30, S31: Bayes dinamis menurut perlakuan heterogenitas ---
s30, s31 = [], []
for wk in CORE:
    for mode in ["fe_unit", "fe_tw", "re", "pooled"]:
        if wk != "queen" and mode in ("re", "pooled") and QUICK:
            continue
        dr = fit_bayes_dyn(Y, ALLW[wk], EV[wk], mode, BAYES_ITER_EMP, BAYES_BURN_EMP, [SEED, 30, CORE.index(wk)])
        sr = np.array([spec_rad(q, EV[wk]) for q in dr[::5]])
        m_, lo, hi = dr.mean(0), np.quantile(dr, .025, axis=0), np.quantile(dr, .975, axis=0)
        s30.append(dict(W=WLAB[wk], effects=mode, tau=m_[1], tau_lwr=lo[1], tau_upr=hi[1], delta=m_[0],
                        eta=m_[2], spectral_radius_mean=sr.mean(), P_stable=(sr < 1).mean()))
        if mode in ("fe_unit", "fe_tw"):
            for nm, a, l_, h_ in zip(PN, m_, lo, hi):
                s31.append(dict(W=WLAB[wk], effects=mode, param=nm, mean=a, lwr95=l_, upr95=h_))
        print(f"Bayes {wk:5s} {mode:8s}: tau {m_[1]:.3f} [{lo[1]:.3f}, {hi[1]:.3f}]")
save(s30, "S30_bayes_dynamic"); save(s31, "S31_bayes_dynamic_coefficients")

In [ ]:
# Sel 11 — Table S33, pemeriksaan otomatis terhadap naskah/addon5, dan paket hasil
from io import StringIO
# S33: degree queen vs impacts tingkat-node (deskriptif)
Wq = ALLW["queen"]; deg = (Wq > 0).sum(1)
s33 = []
for fe in ["ind", "tw"]:
    f = FITS[("SDM", "queen", fe, "full")]; r = f["theta"][0]; js = COL_X.index("sanit")
    S = np.linalg.inv(np.eye(N) - r * Wq) @ (f["theta"][1 + js] * np.eye(N) + f["theta"][1 + K + js] * Wq)
    for nm, v in [("emission (col sum - diag)", S.sum(0) - np.diag(S)), ("reception (row sum - diag)", S.sum(1) - np.diag(S))]:
        rho_s, p_s = spearmanr(deg, v)
        s33.append(dict(FE=fe, quantity=nm, spearman_with_degree=rho_s, p=p_s))
save(s33, "S33_degree_vs_node_impacts")

# ---------------- pemeriksaan ----------------
chk = []
def add(item, computed, reference, tol):
    chk.append(dict(item=item, computed=computed, reference=reference, diff=computed - reference,
                    status="OK" if abs(computed - reference) <= tol else "BEDA"))
g = lambda wk, fe, smp="full": FITS[("SDM", wk, fe, smp)]
add("rho queen unit FE (naskah)", g("queen", "ind")["theta"][0], REF_MS["rho_queen_ind"], 0.005)
add("rho queen two-way FE (naskah)", g("queen", "tw")["theta"][0], REF_MS["rho_queen_tw"], 0.005)
add("rho queen unit 2022-24", g("queen", "ind", "2022-2024")["theta"][0], REF_MS["rho_queen_ind_sub"], 0.005)
add("rho queen two-way 2022-24", g("queen", "tw", "2022-2024")["theta"][0], REF_MS["rho_queen_tw_sub"], 0.005)
add("SE rho queen unit", np.sqrt(g("queen", "ind")["cov"][0, 0]), REF_MS["se_rho_queen_ind"], 0.005)
add("SE rho queen two-way", np.sqrt(g("queen", "tw")["cov"][0, 0]), REF_MS["se_rho_queen_tw"], 0.005)
pv = T2[T2.variable == "poverty"].iloc[0]
add("poverty share left unit (%)", pv.share_left_unit_pct, REF_MS["share_pov_unit_pct"], 0.05)
add("poverty share left two-way (%)", pv.share_left_twoway_pct, REF_MS["share_pov_tw_pct"], 0.01)
add("poverty SD within (Table 2)", pv.sd_within_NTm1, REF_MS["sd_within_pov"], 0.001)
add("year dummies R2 of Wy queen (%)", S3b.iloc[0].R2_year_on_unit_demeaned_Wy_pct, REF_MS["wy_year_R2_pct"], 0.5)
for e in ["total", "direct", "indirect"]:
    v = T4[(T4.FE == "ind") & (T4.variable == "manuf") & (T4.effect == e)].estimate.iloc[0]
    add(f"manuf {e} queen unit FE", v, REF_MS[f"manuf_{e}_queen_ind"], 0.0006)
add("dAIC two-way KNN-4 - queen", FITS[("SDM", "knn4", "tw", "full")]["aic"] - g("queen", "tw")["aic"], REF_MS["dAIC_tw_queen_knn"], 0.06)
add("dAIC two-way econ - queen", FITS[("SDM", "econ", "tw", "full")]["aic"] - g("queen", "tw")["aic"], REF_MS["dAIC_tw_queen_econ"], 0.06)
# addon5: impacts per W (E5b)
e5b = pd.read_csv(StringIO(REF_CSV["E5b"]))
inv = {v: k for k, v in WLAB.items()}
for _, r in e5b.iterrows():
    wk = inv.get(r.W)
    if wk in EXT:
        mine = S8[(S8.W == wk) & (S8.FE == r.FE) & (S8.model == "SDM") & (S8["sample"] == "full")
                  & (S8.variable == r.variable) & (S8.effect == r.effect)].estimate.iloc[0]
        add(f"E5b {wk} {r.FE} {r.variable} {r.effect}", mine, r.estimate, max(0.002, 0.01 * abs(r.estimate)))
# addon5: elpd (E2) dan bobot (E4)
e2 = pd.read_csv(StringIO(REF_CSV["E2"]))
for _, r in e2.iterrows():
    wk = inv.get(r.W)
    sub = S11[(S11.set == r.set) & (S11.FE == r.FE) & (S11.criterion == "loo") & (S11.W == r.W)]
    if wk in EXT and len(sub):
        add(f"elpd_loo {r.set} {r.FE} {wk}", sub.elpd.iloc[0], r.elpd_loo, 1.0)
e4 = pd.read_csv(StringIO(REF_CSV["E4"]))
for _, r in e4.iterrows():
    sub = S13a[(S13a.set == r.set) & (S13a.FE == r.FE) & (S13a.W == r.W)]
    if len(sub):
        for c in ["akaike", "stacking_loo", "bma_P_W_given_SDM"]:
            add(f"{c} {r.set} {r.FE} {inv.get(r.W)}", sub[c].iloc[0], r[c], 0.03)
t01 = pd.read_csv(StringIO(REF_CSV["T01"]))
for _, r in t01.iterrows():
    sub = T1[T1.W == r.W]
    if len(sub):
        for c in ["mean_degree", "min_real_eig", "edge_reciprocity", "clustering", "strong_components"]:
            add(f"T1 {inv.get(r.W)} {c}", float(sub[c].iloc[0]), float(r[c]), 0.001)
# dinamis
q7 = T7[T7.W == WLAB["queen"]].set_index("FE")
add("Table 7 tau queen unit (bootstrap)", q7.loc["ind", "tau"], REF_MS["tau_queen_ind_T7"], 0.02)
add("Table 7 tau queen two-way (bootstrap)", q7.loc["tw", "tau"], REF_MS["tau_queen_tw_T7"], 0.02)
t8 = S29[(S29.W == WLAB["queen"]) & (S29.variable == "sanit")].set_index(["FE", "effect"])
add("Table 8 SR indirect unit", t8.loc[("ind", "SR_indirect"), "estimate"], REF_MS["T8_SR_ind"], 0.01)
add("Table 8 SR indirect two-way", t8.loc[("tw", "SR_indirect"), "estimate"], REF_MS["T8_SR_tw"], 0.01)
add("Table 8 LR indirect unit", t8.loc[("ind", "LR_indirect"), "estimate"], REF_MS["T8_LR_ind"], 0.015)
add("Table 8 LR indirect two-way", t8.loc[("tw", "LR_indirect"), "estimate"], REF_MS["T8_LR_tw"], 0.015)
s30d = pd.DataFrame(s30).set_index(["W", "effects"])
for mode, key in [("fe_unit", "S30_tau_fe_ind"), ("fe_tw", "S30_tau_fe_tw"), ("re", "S30_tau_re"), ("pooled", "S30_tau_pooled")]:
    if (WLAB["queen"], mode) in s30d.index:
        add(f"S30 Bayes tau queen {mode}", s30d.loc[(WLAB["queen"], mode), "tau"], REF_MS[key], 0.02)
if RUN_MC:
    sy = S5[S5.year_effects_in_DGP].set_index("FE")
    add("S5 mean rho unit FE (DGP with year effects)", sy.loc["ind", "mean_rho"], REF_MS["S5_rho_unit_mean"], 0.03)
    add("S5 mean rho two-way FE", sy.loc["tw", "mean_rho"], REF_MS["S5_rho_tw_mean"], 0.03)
    for w_, key in [("queen", "S32_AIC_queen"), ("knn4", "S32_AIC_knn4"), ("econ", "S32_AIC_econ")]:
        add(f"S32 AIC recovery {w_}", S32.set_index("true_W").loc[w_, "AIC_correct"], REF_MS[key], 0.04)

CHK = save(chk, "CHECK_vs_manuscript_and_addon5")
print(f"\nPEMERIKSAAN: {(CHK.status == 'OK').sum()} OK, {(CHK.status == 'BEDA').sum()} BEDA dari {len(CHK)} item")
pd.set_option("display.max_rows", 400, "display.width", 200)
print(CHK[CHK.status == "BEDA"].round(4).to_string(index=False) if (CHK.status == "BEDA").any() else "Semua cocok.")
print(CHK[~CHK.item.str.startswith(("E5b", "elpd", "akaike", "stacking", "bma", "T1 "))].round(4).to_string(index=False))

import shutil
shutil.make_archive(os.path.join(OUT, "nssp_main_results"), "zip", RES)
print("\nSemua tabel tersimpan di", RES, "dan nssp_main_results.zip")